In [ ]:
# Optional Kaggle-only recovery command from the original development session.
# !kaggle kernels output afsanatusa05/notebook75e9ef4942 -p /kaggle/working/vit_outputs
print("Skipping optional Kaggle kernel-output download. Use existing checkpoints if required.")


In [ ]:
# Optional inspection of a previously downloaded Kaggle log.
log_path = os.path.join(OUTPUT_ROOT, "vit_outputs", "notebook75e9ef4942.log")
if os.path.exists(log_path):
    with open(log_path, "r", errors="ignore") as f:
        log = f.readlines()
    keywords = ["best_vit.pth", "PURE VIT", "Saved:", "Traceback", "Error"]
    for i, line in enumerate(log):
        if any(word.lower() in line.lower() for word in keywords):
            print(f"Line {i+1}: {line.rstrip()}")
else:
    print("No optional ViT recovery log found; continuing with the notebook workflow.")


In [ ]:
# Search configured dataset/output folders for existing ViT checkpoints.
search_dirs = [DATASET_ROOT, OUTPUT_ROOT]
print("=== Searching for ViT checkpoints ===")
for base_dir in search_dirs:
    if os.path.exists(base_dir):
        for root, dirs, files in os.walk(base_dir):
            for file in files:
                name = file.lower()
                if ("vit" in name or "vision" in name) and name.endswith((".pth", ".pt", ".ckpt")):
                    print(os.path.join(root, file))


In [ ]:
# List files in the optional ViT recovery-output folder.
vit_output_dir = os.path.join(OUTPUT_ROOT, "vit_outputs")
if os.path.isdir(vit_output_dir):
    for root, dirs, files in os.walk(vit_output_dir):
        for file in files:
            print(os.path.join(root, file))
else:
    print("No optional ViT recovery-output folder found.")


In [ ]:
# ============================================================
# 🌿 LEAF DETECTION RESEARCH
# Notebook 0: Dataset Audit & Metadata Generation
# ============================================================

import os
import glob
import json
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

print("Environment ready!")

# ============================================================
# REPRODUCIBILITY CONFIGURATION
# ============================================================
# Set LEAF_DATASET_ROOT to the directory containing the two source
# dataset folders. In Kaggle, the default below matches the original
# research environment. For local use, set the environment variable.

import os
from pathlib import Path

DATASET_ROOT = os.environ.get(
    "LEAF_DATASET_ROOT",
    "/kaggle/input/datasets/afsanatusa05/leaf-detection"
)
DATASET_PARENT = os.path.dirname(DATASET_ROOT.rstrip(os.sep))
OUTPUT_ROOT = os.environ.get("LEAF_OUTPUT_DIR", "./outputs")
os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Dataset root:", DATASET_ROOT)
print("Output root:", OUTPUT_ROOT)


In [ ]:
# List everything inside /kaggle/input

for root, dirs, files in os.walk(DATASET_PARENT):
    level = root.replace(DATASET_PARENT, "").count(os.sep)
    
    if level <= 2:
        print("  " * level + os.path.basename(root) + "/")

In [ ]:
# ============================================================
# Inspect the dataset directory
# ============================================================

dataset_root = DATASET_PARENT

print("Dataset root:", dataset_root)
print("\nContents:\n")

for item in os.listdir(dataset_root):
    path = os.path.join(dataset_root, item)
    
    if os.path.isdir(path):
        print(f"[DIR]  {item}")
    else:
        print(f"[FILE] {item}")

In [ ]:
# Show folder structure up to 3 levels

for root, dirs, files in os.walk(dataset_root):
    level = root.replace(dataset_root, "").count(os.sep)
    
    if level <= 3:
        indent = "    " * level
        print(f"{indent}{os.path.basename(root)}/")

In [ ]:
# ============================================================
# CELL 04: Inspect All Dataset Classes
# ============================================================

for root, dirs, files in os.walk(dataset_root):
    level = root.replace(dataset_root, "").count(os.sep)
    
    if level <= 5:
        indent = "    " * level
        print(f"{indent}{os.path.basename(root)}/")


In [ ]:
# ============================================================
# CELL 05: Count Images in Each Class Folder
# ============================================================

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

folder_stats = []

for root, dirs, files in os.walk(dataset_root):
    image_files = [
        f for f in files
        if Path(f).suffix.lower() in IMAGE_EXTENSIONS
    ]
    
    if image_files:
        folder_stats.append({
            "folder": os.path.basename(root),
            "path": root,
            "image_count": len(image_files)
        })

folder_stats_df = pd.DataFrame(folder_stats)

folder_stats_df = folder_stats_df.sort_values(
    by="image_count",
    ascending=False
).reset_index(drop=True)

print(f"Total folders containing images: {len(folder_stats_df)}")
print(f"Total images: {folder_stats_df['image_count'].sum()}")

display(folder_stats_df)

In [ ]:
# ============================================================
# CELL 06: Image Format and Empty Folder Analysis
# ============================================================

# Count image extensions
extension_counter = Counter()

for root, dirs, files in os.walk(dataset_root):
    for file in files:
        ext = Path(file).suffix.lower()
        
        if ext in IMAGE_EXTENSIONS:
            extension_counter[ext] += 1

print("Image format distribution:")
for ext, count in sorted(extension_counter.items()):
    print(f"{ext:8s} : {count}")

# Find empty folders
empty_folders = []

for root, dirs, files in os.walk(dataset_root):
    has_images = any(
        Path(f).suffix.lower() in IMAGE_EXTENSIONS
        for f in files
    )
    
    if not dirs and not has_images:
        empty_folders.append(root)

print("\nNumber of empty folders:", len(empty_folders))

if empty_folders:
    print("\nEmpty folders:")
    for folder in empty_folders:
        print(folder)

In [ ]:
# ============================================================
# CELL 07: Build Master Folder Audit Table
# ============================================================

folder_records = []

for root, dirs, files in os.walk(dataset_root):
    image_files = [
        f for f in files
        if Path(f).suffix.lower() in IMAGE_EXTENSIONS
    ]
    
    if not image_files:
        continue

    relative_path = os.path.relpath(root, dataset_root)
    path_parts = Path(relative_path).parts
    
    # Identify source dataset
    if "Our research pictures" in path_parts:
        source = "Our Research Pictures"
    elif "Medical plants" in path_parts:
        source = "Medical Plants"
    else:
        source = "Unknown Source"

    folder_name = os.path.basename(root)

    folder_records.append({
        "source": source,
        "folder_name": folder_name,
        "image_count": len(image_files),
        "folder_path": root
    })

folder_audit_df = pd.DataFrame(folder_records)

folder_audit_df = folder_audit_df.sort_values(
    by=["source", "image_count"],
    ascending=[True, False]
).reset_index(drop=True)

print("Total image-containing folders:", len(folder_audit_df))
print("Total images:", folder_audit_df["image_count"].sum())

display(folder_audit_df)

In [ ]:
# ============================================================
# CELL 08: Source-wise Dataset Statistics
# ============================================================

source_summary = (
    folder_audit_df
    .groupby("source")
    .agg(
        number_of_folders=("folder_name", "count"),
        total_images=("image_count", "sum"),
        average_images_per_folder=("image_count", "mean"),
        minimum_images=("image_count", "min"),
        maximum_images=("image_count", "max")
    )
    .reset_index()
)

display(source_summary)

In [ ]:
# ============================================================
# CELL 09: Detect Duplicate Folder Names Across Sources
# ============================================================

duplicate_folder_names = (
    folder_audit_df
    .groupby("folder_name")["source"]
    .nunique()
)

duplicate_folder_names = duplicate_folder_names[
    duplicate_folder_names > 1
]

print("Folder names appearing in multiple sources:")
print()

if len(duplicate_folder_names) == 0:
    print("No duplicate folder names found across sources.")
else:
    for folder_name in duplicate_folder_names.index:
        print(f"• {folder_name}")
        
        matching_rows = folder_audit_df[
            folder_audit_df["folder_name"] == folder_name
        ]
        
        for _, row in matching_rows.iterrows():
            print(
                f"    {row['source']}: "
                f"{row['image_count']} images"
            )

In [ ]:
# ============================================================
# CELL 10: Raw Folder Inventory
# ============================================================

raw_inventory = folder_audit_df[
    ["source", "folder_name", "image_count"]
].copy()

raw_inventory = raw_inventory.sort_values(
    by=["source", "folder_name"]
).reset_index(drop=True)

print("Complete Raw Folder Inventory")
print("=" * 70)

display(raw_inventory)

In [ ]:
# ============================================================
# CELL 11: Preliminary Health Label Detection
# ============================================================

def detect_health(folder_name):
    name = folder_name.lower().strip()
    
    if name.startswith("healthy"):
        return "Healthy"
    
    if name.startswith("disease"):
        return "Diseased"
    
    return "Unknown"


health_audit = folder_audit_df.copy()

health_audit["preliminary_health"] = (
    health_audit["folder_name"]
    .apply(detect_health)
)

health_summary = (
    health_audit
    .groupby(["source", "preliminary_health"])
    .agg(
        folders=("folder_name", "count"),
        images=("image_count", "sum")
    )
    .reset_index()
)

display(health_summary)

In [ ]:
# ============================================================
# CELL 12: Create Label Mapping Template
# ============================================================

label_mapping = folder_audit_df[
    ["source", "folder_name", "image_count"]
].copy()

# Preliminary health label
label_mapping["health"] = (
    label_mapping["folder_name"]
    .apply(detect_health)
)

# These columns will be completed after manual taxonomy verification
label_mapping["species"] = "TO_BE_MAPPED"
label_mapping["category"] = "TO_BE_MAPPED"

label_mapping = label_mapping[
    [
        "source",
        "folder_name",
        "species",
        "category",
        "health",
        "image_count"
    ]
]

display(label_mapping)

In [ ]:
# ============================================================
# CELL 13: Unique Folder Names for Taxonomy Mapping
# ============================================================

unique_folders = (
    label_mapping[
        ["source", "folder_name", "image_count"]
    ]
    .sort_values(["source", "folder_name"])
    .reset_index(drop=True)
)

display(unique_folders)

In [ ]:
# ============================================================
# CELL 14: Species Name Normalization
# ============================================================

# Normalize raw folder names first
label_mapping["folder_name_clean"] = (
    label_mapping["folder_name"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

species_mapping = {

    # -----------------------------
    # Medical Plants
    # -----------------------------
    "Amloki": "Amloki",
    "Orjun": "Arjun",
    "Patorkusi": "Patorkusi",
    "alovera": "Aloe vera",
    "আদা": "Ginger",
    "কালোজিরা": "Black cumin",
    "তুলশী গাছ": "Tulsi",
    "নিম পাতা": "Neem",
    "বাসক": "Vasaka",
    "সজনা গাছ": "Moringa",
    "হলুদ গাছ": "Turmeric",

    # -----------------------------
    # Our Research Pictures
    # -----------------------------
    "healthy kochu pataa": "Kochu",
    "Disease kosu leaf": "Kochu",

    "Healthy বড়ই (boroi)": "Boroi",
    "disease বড়ই (boroi)": "Boroi",

    "Almond Tree": "Almond",
    "Mango tree": "Mango",

    "disease sapodilla(sofeda)": "Sapodilla",
    "healthy sapodilla (sofeda)": "Sapodilla",

    "disease Jacfruits": "Jackfruit",
    "Healthy Jackfruits": "Jackfruit",
    "Jackfruit": "Jackfruit",

    "Healthy banana leaf": "Banana",
    "Disease Banana leaf": "Banana",

    "disease puishak": "Puishak",
    "healthy puishak": "Puishak",

    "disease beal leaf": "Bael",
    "healthy beal leaf": "Bael",

    "Disease Green chili": "Green chili",
    "healthy green chilli": "Green chili",

    "healthy lichi (লিচু)": "Litchi",
    "Disease lichi (লিচু )": "Litchi",

    "Chili or Chilli pepper(Morich)": "Chili",

    "Apple": "Apple",
    "Guava": "Guava",
    "পেঁপে (Papaya)": "Papaya",
    "Potato আলু": "Potato",
    "Lemon": "Lemon",

    "Pomegranate (ডালিম বেদানা)": "Pomegranate",

    # -----------------------------
    # Flower Plants
    # -----------------------------
    "Marigold": "Marigold",
    "Chrysanthemum": "Chrysanthemum",

    # -----------------------------
    # Other Trees
    # -----------------------------
    "Bakul tree": "Bakul",
    "Peepal tree": "Peepal",
}

# Apply normalized mapping
label_mapping["species"] = (
    label_mapping["folder_name_clean"]
    .map(species_mapping)
)

# Check unmapped folders
unmapped = (
    label_mapping[
        label_mapping["species"].isna()
    ]["folder_name_clean"]
    .unique()
)

print("Total folders:", len(label_mapping))
print(
    "Mapped folders:",
    label_mapping["species"].notna().sum()
)
print("Unmapped folders:", len(unmapped))

if len(unmapped) > 0:
    print("\nUnmapped folders:")
    for folder in unmapped:
        print(" -", folder)
else:
    print("\nAll folders mapped successfully! ✓")

display(
    label_mapping[
        [
            "source",
            "folder_name",
            "species",
            "category",
            "health",
            "image_count"
        ]
    ]
)

In [ ]:
# ============================================================
# CELL 15: Finalize Health Labels
# ============================================================

# Based on the original data-collection protocol,
# folders without an explicit disease label are treated as Healthy.

label_mapping["health"] = (
    label_mapping["health"]
    .replace("Unknown", "Healthy")
)

print("Final health label distribution:")
print("=" * 50)

health_counts = (
    label_mapping
    .groupby("health")["image_count"]
    .sum()
    .sort_values(ascending=False)
)

display(health_counts.to_frame("image_count"))

In [ ]:
# ============================================================
# CELL 16: Plant Category Mapping
# ============================================================

category_mapping = {

    # --------------------------------------------------------
    # MEDICINAL PLANTS
    # --------------------------------------------------------
    "Aloe vera": "Medicinal",
    "Tulsi": "Medicinal",
    "Vasaka": "Medicinal",
    "Neem": "Medicinal",
    "Moringa": "Medicinal",
    "Turmeric": "Medicinal",
    "Amloki": "Medicinal",
    "Patorkusi": "Medicinal",
    "Black cumin": "Medicinal",
    "Arjun": "Medicinal",
    "Ginger": "Medicinal",

    # --------------------------------------------------------
    # FRUIT PLANTS
    # --------------------------------------------------------
    "Boroi": "Fruit",
    "Almond": "Fruit",
    "Mango": "Fruit",
    "Sapodilla": "Fruit",
    "Jackfruit": "Fruit",
    "Banana": "Fruit",
    "Litchi": "Fruit",
    "Apple": "Fruit",
    "Guava": "Fruit",
    "Papaya": "Fruit",
    "Lemon": "Fruit",
    "Pomegranate": "Fruit",
    "Bael": "Fruit",

    # --------------------------------------------------------
    # VEGETABLE / CROP PLANTS
    # --------------------------------------------------------
    "Kochu": "Vegetable/Crop",
    "Puishak": "Vegetable/Crop",
    "Chili": "Vegetable/Crop",
    "Green chili": "Vegetable/Crop",
    "Potato": "Vegetable/Crop",

    # --------------------------------------------------------
    # FLOWER / ORNAMENTAL PLANTS
    # --------------------------------------------------------
    "Marigold": "Flower/Ornamental",
    "Chrysanthemum": "Flower/Ornamental",

    # --------------------------------------------------------
    # OTHER TREES
    # --------------------------------------------------------
    "Bakul": "Flower/Ornamental",
    "Peepal": "Medicinal",
}


# Apply category mapping
label_mapping["category"] = (
    label_mapping["species"].map(category_mapping)
)


# Check mapping completeness
unmapped_categories = (
    label_mapping[
        label_mapping["category"].isna()
    ]["species"]
    .unique()
)


print("Total folders:", len(label_mapping))
print(
    "Mapped categories:",
    label_mapping["category"].notna().sum()
)
print(
    "Unmapped categories:",
    len(unmapped_categories)
)

if len(unmapped_categories) > 0:
    print("\nSpecies without category:")
    for species in unmapped_categories:
        print(" -", species)
else:
    print("\nAll species categorized successfully! ✓")


# Display final label table
display(
    label_mapping[
        [
            "source",
            "folder_name",
            "species",
            "category",
            "health",
            "image_count"
        ]
    ]
)

In [ ]:
# ============================================================
# CELL 17: Category Distribution
# ============================================================

category_counts = (
    label_mapping
    .groupby("category")["image_count"]
    .sum()
    .sort_values(ascending=False)
)

print("Final category distribution:")
print("=" * 50)

display(
    category_counts.to_frame(name="image_count")
)

print("\nTotal images:", category_counts.sum())
print("Total categories:", len(category_counts))

In [ ]:
# ============================================================
# CELL 18: Species-wise Dataset Distribution
# ============================================================

species_counts = (
    label_mapping
    .groupby("species")["image_count"]
    .sum()
    .sort_values(ascending=False)
)

print("Species-wise image distribution:")
print("=" * 50)

display(
    species_counts.to_frame(name="image_count")
)

print("\nTotal species:", len(species_counts))
print("Total images:", species_counts.sum())

print("\nSmallest species classes:")
display(
    species_counts
    .sort_values()
    .head(10)
    .to_frame(name="image_count")
)

print("\nLargest species classes:")
display(
    species_counts
    .sort_values(ascending=False)
    .head(10)
    .to_frame(name="image_count")
)

In [ ]:
# ============================================================
# CELL 19: Species-wise Health Distribution
# ============================================================

species_health_distribution = (
    label_mapping
    .pivot_table(
        index="species",
        columns="health",
        values="image_count",
        aggfunc="sum",
        fill_value=0
    )
)

# Ensure both health columns exist
for col in ["Healthy", "Diseased"]:
    if col not in species_health_distribution.columns:
        species_health_distribution[col] = 0

# Keep a consistent column order
species_health_distribution = species_health_distribution[
    ["Healthy", "Diseased"]
]

# Add total images per species
species_health_distribution["Total"] = (
    species_health_distribution["Healthy"]
    + species_health_distribution["Diseased"]
)

# Sort by total images
species_health_distribution = (
    species_health_distribution
    .sort_values("Total", ascending=False)
)

print("Species-wise health distribution:")
print("=" * 60)

display(species_health_distribution)

print("\nTotal Healthy:",
      species_health_distribution["Healthy"].sum())

print("Total Diseased:",
      species_health_distribution["Diseased"].sum())

print("Total Images:",
      species_health_distribution["Total"].sum())

In [ ]:
# ============================================================
# CELL 20: Image Quality Audit
# ============================================================

from PIL import Image
import os
import pandas as pd

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".tif", ".tiff", ".webp"
}

BASE_PATH = DATASET_ROOT

quality_records = []
corrupted_images = []
missing_folders = []


# ------------------------------------------------------------
# Scan all mapped folders
# ------------------------------------------------------------

for _, row in label_mapping.iterrows():

    source = row["source"]
    folder_name = row["folder_name"]

    # Find source directory
    source_candidates = [
        os.path.join(BASE_PATH, source),
        os.path.join(
            BASE_PATH,
            "Our Research Pictures-20260712T062632Z-2-001",
            "Our Research Pictures"
        ) if source == "Our Research Pictures" else None,
        os.path.join(
            BASE_PATH,
            "Medical plants-20260712T062633Z-2-001",
            "Medical plants"
        ) if source == "Medical Plants" else None
    ]

    source_path = None

    for candidate in source_candidates:

        if candidate is not None and os.path.isdir(candidate):
            source_path = candidate
            break

    if source_path is None:
        missing_folders.append(
            f"Source not found: {source}"
        )
        continue


    # Actual image folder
    folder_path = os.path.join(
        source_path,
        folder_name
    )

    if not os.path.isdir(folder_path):

        missing_folders.append(
            folder_path
        )

        continue


    # --------------------------------------------------------
    # Scan images
    # --------------------------------------------------------

    for file_name in os.listdir(folder_path):

        file_path = os.path.join(
            folder_path,
            file_name
        )

        ext = os.path.splitext(
            file_name
        )[1].lower()

        if ext not in IMAGE_EXTENSIONS:
            continue

        try:

            # Check image integrity
            with Image.open(file_path) as img:
                img.verify()


            # Read image properties
            with Image.open(file_path) as img:

                width, height = img.size
                mode = img.mode


            quality_records.append({

                "file_path": file_path,
                "source": source,
                "folder_name": folder_name,
                "species": row["species"],
                "category": row["category"],
                "health": row["health"],
                "width": width,
                "height": height,
                "mode": mode,
                "extension": ext

            })


        except Exception as e:

            corrupted_images.append({

                "file_path": file_path,
                "error": str(e)

            })


# ------------------------------------------------------------
# Create DataFrames
# ------------------------------------------------------------

quality_df = pd.DataFrame(
    quality_records
)

corrupted_df = pd.DataFrame(
    corrupted_images
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("IMAGE QUALITY AUDIT")
print("=" * 60)

print(
    "Expected images :",
    label_mapping["image_count"].sum()
)

print(
    "Readable images :",
    len(quality_df)
)

print(
    "Corrupted images:",
    len(corrupted_df)
)

print(
    "Missing folders :",
    len(missing_folders)
)


# ------------------------------------------------------------
# Image modes
# ------------------------------------------------------------

if len(quality_df) > 0:

    print("\nImage modes:")
    print("-" * 40)

    print(
        quality_df["mode"].value_counts()
    )


# ------------------------------------------------------------
# File extensions
# ------------------------------------------------------------

if len(quality_df) > 0:

    print("\nFile extensions:")
    print("-" * 40)

    print(
        quality_df["extension"].value_counts()
    )


# ------------------------------------------------------------
# Image dimensions
# ------------------------------------------------------------

if len(quality_df) > 0:

    print("\nImage width statistics:")
    print("-" * 40)

    print(
        quality_df["width"].describe()
    )

    print("\nImage height statistics:")
    print("-" * 40)

    print(
        quality_df["height"].describe()
    )


# ------------------------------------------------------------
# Missing folders
# ------------------------------------------------------------

if len(missing_folders) > 0:

    print("\nMissing folders:")
    print("-" * 40)

    for folder in missing_folders:
        print(folder)


# ------------------------------------------------------------
# Corrupted images
# ------------------------------------------------------------

if len(corrupted_df) > 0:

    print("\nCorrupted images:")
    print("-" * 40)

    display(corrupted_df)

else:

    print("\nNo corrupted images detected! ✓")

In [ ]:
# ============================================================
# CELL 20A: Inspect Label Mapping Structure
# ============================================================

print("label_mapping columns:")
print("=" * 60)

print(label_mapping.columns.tolist())

print("\nFirst 5 rows:")
display(label_mapping.head())

In [ ]:
# ============================================================
# CELL 20B: Inspect Actual Dataset Structure
# ============================================================

import os

BASE_PATH = DATASET_ROOT

print("BASE PATH:")
print(BASE_PATH)

print("\nTop-level contents:")
print("=" * 60)

for item in os.listdir(BASE_PATH):
    full_path = os.path.join(BASE_PATH, item)

    if os.path.isdir(full_path):
        print("📁", repr(item))
    else:
        print("📄", repr(item))

In [ ]:
# ============================================================
# CELL 20C: Inspect Dataset Subdirectories
# ============================================================

print("DATASET SUBDIRECTORIES")
print("=" * 60)

for item in os.listdir(BASE_PATH):

    item_path = os.path.join(BASE_PATH, item)

    if os.path.isdir(item_path):

        print(f"\n📁 {item}")

        for subitem in os.listdir(item_path):

            sub_path = os.path.join(item_path, subitem)

            if os.path.isdir(sub_path):
                print(f"   📁 {subitem}")
            else:
                print(f"   📄 {subitem}")

In [ ]:
# ============================================================
# CELL 20D: Final Image Quality Audit
# ============================================================

from PIL import Image
import os
import pandas as pd

# ------------------------------------------------------------
# Dataset paths
# ------------------------------------------------------------

BASE_PATH = DATASET_ROOT

SOURCE_PATHS = {
    "Our Research Pictures": os.path.join(
        BASE_PATH,
        "Our research pictures-20260712T062632Z-2-001",
        "Our research pictures"
    ),

    "Medical Plants": os.path.join(
        BASE_PATH,
        "Medical plants-20260712T062633Z-2-001",
        "Medical plants"
    )
}


# ------------------------------------------------------------
# Supported image formats
# ------------------------------------------------------------

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp"
}


# ------------------------------------------------------------
# Storage
# ------------------------------------------------------------

quality_records = []
corrupted_images = []
missing_folders = []


# ------------------------------------------------------------
# Scan all 44 mapped folders
# ------------------------------------------------------------

for _, row in label_mapping.iterrows():

    source = row["source"]
    folder_name = row["folder_name"]


    # Get correct parent directory
    source_path = SOURCE_PATHS.get(source)


    if source_path is None:

        missing_folders.append(
            f"Unknown source: {source}"
        )

        continue


    # Build exact folder path
    folder_path = os.path.join(
        source_path,
        folder_name
    )


    # Check folder existence
    if not os.path.isdir(folder_path):

        missing_folders.append(
            folder_path
        )

        continue


    # --------------------------------------------------------
    # Scan images
    # --------------------------------------------------------

    for file_name in os.listdir(folder_path):

        file_path = os.path.join(
            folder_path,
            file_name
        )

        ext = os.path.splitext(
            file_name
        )[1].lower()


        if ext not in IMAGE_EXTENSIONS:
            continue


        try:

            # Verify image integrity
            with Image.open(file_path) as img:
                img.verify()


            # Read image properties
            with Image.open(file_path) as img:

                width, height = img.size
                mode = img.mode


            quality_records.append({

                "file_path": file_path,
                "source": source,
                "folder_name": folder_name,
                "species": row["species"],
                "category": row["category"],
                "health": row["health"],
                "width": width,
                "height": height,
                "mode": mode,
                "extension": ext

            })


        except Exception as e:

            corrupted_images.append({

                "file_path": file_path,
                "error": str(e)

            })


# ------------------------------------------------------------
# Create DataFrames
# ------------------------------------------------------------

quality_df = pd.DataFrame(
    quality_records
)

corrupted_df = pd.DataFrame(
    corrupted_images
)


# ------------------------------------------------------------
# Audit summary
# ------------------------------------------------------------

expected_images = int(
    label_mapping["image_count"].sum()
)

readable_images = len(
    quality_df
)

corrupted_count = len(
    corrupted_df
)

missing_count = len(
    missing_folders
)


print("IMAGE QUALITY AUDIT")
print("=" * 60)

print(
    f"Expected images : {expected_images}"
)

print(
    f"Readable images : {readable_images}"
)

print(
    f"Corrupted images: {corrupted_count}"
)

print(
    f"Missing folders : {missing_count}"
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("\nAUDIT STATUS")
print("=" * 60)

if (
    expected_images == readable_images
    and corrupted_count == 0
    and missing_count == 0
):

    print("✓ All 1299 images verified successfully!")

else:

    print("⚠ Dataset audit requires further inspection.")


# ------------------------------------------------------------
# Image modes
# ------------------------------------------------------------

if len(quality_df) > 0:

    print("\nImage modes:")
    print("-" * 40)

    print(
        quality_df["mode"].value_counts()
    )


# ------------------------------------------------------------
# File extensions
# ------------------------------------------------------------

if len(quality_df) > 0:

    print("\nFile extensions:")
    print("-" * 40)

    print(
        quality_df["extension"].value_counts()
    )


# ------------------------------------------------------------
# Image dimensions
# ------------------------------------------------------------

if len(quality_df) > 0:

    print("\nImage width statistics:")
    print("-" * 40)

    print(
        quality_df["width"].describe()
    )


    print("\nImage height statistics:")
    print("-" * 40)

    print(
        quality_df["height"].describe()
    )


# ------------------------------------------------------------
# Missing folders
# ------------------------------------------------------------

if missing_count > 0:

    print("\nMissing folders:")
    print("-" * 40)

    for folder in missing_folders:
        print(folder)


# ------------------------------------------------------------
# Corrupted images
# ------------------------------------------------------------

if corrupted_count > 0:

    print("\nCorrupted images:")
    print("-" * 40)

    display(corrupted_df)

In [ ]:
# ============================================================
# CELL 21: Duplicate & Near-Duplicate Image Detection
# ============================================================

import hashlib
from PIL import Image
import numpy as np
from collections import defaultdict


# ------------------------------------------------------------
# Exact Duplicate Detection using MD5
# ------------------------------------------------------------

print("EXACT DUPLICATE IMAGE CHECK")
print("=" * 60)

md5_groups = defaultdict(list)

for file_path in quality_df["file_path"]:

    try:
        with open(file_path, "rb") as f:
            file_hash = hashlib.md5(f.read()).hexdigest()

        md5_groups[file_hash].append(file_path)

    except Exception as e:
        print(f"Error reading: {file_path}")


exact_duplicate_groups = {
    h: paths
    for h, paths in md5_groups.items()
    if len(paths) > 1
}


print(
    "Unique image files :",
    len(md5_groups)
)

print(
    "Duplicate groups    :",
    len(exact_duplicate_groups)
)

print(
    "Images involved     :",
    sum(len(v) for v in exact_duplicate_groups.values())
)


# ------------------------------------------------------------
# Show exact duplicates
# ------------------------------------------------------------

if len(exact_duplicate_groups) > 0:

    print("\nExact duplicate groups:")
    print("-" * 60)

    for i, paths in enumerate(
        exact_duplicate_groups.values(),
        start=1
    ):

        print(f"\nGroup {i}:")

        for path in paths:
            print("  ", path)

else:

    print("\nNo exact duplicate images detected! ✓")


# ------------------------------------------------------------
# Perceptual Hash for Near-Duplicate Detection
# ------------------------------------------------------------

print("\n\nNEAR-DUPLICATE IMAGE CHECK")
print("=" * 60)

def average_hash(image_path, hash_size=16):

    with Image.open(image_path) as img:

        img = img.convert("L")
        img = img.resize(
            (hash_size, hash_size)
        )

        pixels = np.asarray(img)

    avg = pixels.mean()

    return (
        pixels > avg
    ).flatten()


def hamming_distance(hash1, hash2):

    return np.count_nonzero(
        hash1 != hash2
    )


# ------------------------------------------------------------
# Generate perceptual hashes
# ------------------------------------------------------------

perceptual_hashes = {}

for file_path in quality_df["file_path"]:

    try:

        perceptual_hashes[file_path] = (
            average_hash(file_path)
        )

    except Exception as e:

        print(
            f"Hashing error: {file_path}"
        )


# ------------------------------------------------------------
# Compare images
# ------------------------------------------------------------

near_duplicate_pairs = []

paths = list(
    perceptual_hashes.keys()
)

for i in range(len(paths)):

    hash1 = perceptual_hashes[
        paths[i]
    ]

    for j in range(i + 1, len(paths)):

        hash2 = perceptual_hashes[
            paths[j]
        ]

        distance = hamming_distance(
            hash1,
            hash2
        )

        # Threshold for possible near duplicates
        if distance <= 8:

            near_duplicate_pairs.append({

                "image_1": paths[i],
                "image_2": paths[j],
                "hamming_distance": distance

            })


near_duplicate_df = pd.DataFrame(
    near_duplicate_pairs
)


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print(
    "Images hashed:",
    len(perceptual_hashes)
)

print(
    "Potential near-duplicate pairs:",
    len(near_duplicate_df)
)


if len(near_duplicate_df) > 0:

    print("\nPotential near-duplicates:")
    print("-" * 60)

    display(
        near_duplicate_df.sort_values(
            "hamming_distance"
        ).head(50)
    )

else:

    print(
        "\nNo potential near-duplicates detected! ✓"
    )


In [ ]:
# ============================================================
# CELL 21: Duplicate & Near-Duplicate Image Detection
# ============================================================

import hashlib
from PIL import Image
import numpy as np
from collections import defaultdict


# ------------------------------------------------------------
# Exact Duplicate Detection using MD5
# ------------------------------------------------------------

print("EXACT DUPLICATE IMAGE CHECK")
print("=" * 60)

md5_groups = defaultdict(list)

for file_path in quality_df["file_path"]:

    try:
        with open(file_path, "rb") as f:
            file_hash = hashlib.md5(f.read()).hexdigest()

        md5_groups[file_hash].append(file_path)

    except Exception as e:
        print(f"Error reading: {file_path}")


exact_duplicate_groups = {
    h: paths
    for h, paths in md5_groups.items()
    if len(paths) > 1
}


print(
    "Unique image files :",
    len(md5_groups)
)

print(
    "Duplicate groups    :",
    len(exact_duplicate_groups)
)

print(
    "Images involved     :",
    sum(len(v) for v in exact_duplicate_groups.values())
)


# ------------------------------------------------------------
# Show exact duplicates
# ------------------------------------------------------------

if len(exact_duplicate_groups) > 0:

    print("\nExact duplicate groups:")
    print("-" * 60)

    for i, paths in enumerate(
        exact_duplicate_groups.values(),
        start=1
    ):

        print(f"\nGroup {i}:")

        for path in paths:
            print("  ", path)

else:

    print("\nNo exact duplicate images detected! ✓")


# ------------------------------------------------------------
# Perceptual Hash for Near-Duplicate Detection
# ------------------------------------------------------------

print("\n\nNEAR-DUPLICATE IMAGE CHECK")
print("=" * 60)

def average_hash(image_path, hash_size=16):

    with Image.open(image_path) as img:

        img = img.convert("L")
        img = img.resize(
            (hash_size, hash_size)
        )

        pixels = np.asarray(img)

    avg = pixels.mean()

    return (
        pixels > avg
    ).flatten()


def hamming_distance(hash1, hash2):

    return np.count_nonzero(
        hash1 != hash2
    )


# ------------------------------------------------------------
# Generate perceptual hashes
# ------------------------------------------------------------

perceptual_hashes = {}

for file_path in quality_df["file_path"]:

    try:

        perceptual_hashes[file_path] = (
            average_hash(file_path)
        )

    except Exception as e:

        print(
            f"Hashing error: {file_path}"
        )


# ------------------------------------------------------------
# Compare images
# ------------------------------------------------------------

near_duplicate_pairs = []

paths = list(
    perceptual_hashes.keys()
)

for i in range(len(paths)):

    hash1 = perceptual_hashes[
        paths[i]
    ]

    for j in range(i + 1, len(paths)):

        hash2 = perceptual_hashes[
            paths[j]
        ]

        distance = hamming_distance(
            hash1,
            hash2
        )

        # Threshold for possible near duplicates
        if distance <= 8:

            near_duplicate_pairs.append({

                "image_1": paths[i],
                "image_2": paths[j],
                "hamming_distance": distance

            })


near_duplicate_df = pd.DataFrame(
    near_duplicate_pairs
)


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print(
    "Images hashed:",
    len(perceptual_hashes)
)

print(
    "Potential near-duplicate pairs:",
    len(near_duplicate_df)
)


if len(near_duplicate_df) > 0:

    print("\nPotential near-duplicates:")
    print("-" * 60)

    display(
        near_duplicate_df.sort_values(
            "hamming_distance"
        ).head(50)
    )

else:

    print(
        "\nNo potential near-duplicates detected! ✓"
    )

In [ ]:
# ============================================================
# CELL 22: CREATE FINAL IMAGE-LEVEL DATASET
# ============================================================

import os
import pandas as pd

# Dataset base path
# DATASET_ROOT is defined in the configuration cell above.

records = []

for _, row in label_mapping.iterrows():

    source = row["source"]
    folder_name = row["folder_name"]

    # --------------------------------------------------------
    # Construct actual folder path
    # --------------------------------------------------------

    if source == "Our Research Pictures":

        folder_path = os.path.join(
            DATASET_ROOT,
            "Our research pictures-20260712T062632Z-2-001",
            "Our research pictures",
            folder_name
        )

    elif source == "Medical Plants":

        folder_path = os.path.join(
            DATASET_ROOT,
            "Medical plants-20260712T062633Z-2-001",
            "Medical plants",
            folder_name
        )

    else:
        continue

    # --------------------------------------------------------
    # Check folder
    # --------------------------------------------------------

    if not os.path.isdir(folder_path):
        print(f"Warning: Folder not found -> {folder_path}")
        continue

    # --------------------------------------------------------
    # Collect images
    # --------------------------------------------------------

    for filename in os.listdir(folder_path):

        if filename.lower().endswith((".jpg", ".jpeg", ".png")):

            image_path = os.path.join(folder_path, filename)

            records.append({
                "image_path": image_path,
                "species": row["species"],
                "category": row["category"],
                "health": row["health"],
                "source": row["source"],
                "folder_name": row["folder_name"]
            })


# ============================================================
# CREATE FINAL DATAFRAME
# ============================================================

image_df = pd.DataFrame(records)

image_df = image_df.reset_index(drop=True)


# ============================================================
# DATASET SUMMARY
# ============================================================

print("FINAL IMAGE DATASET")
print("=" * 60)

print(f"Total images    : {len(image_df)}")
print(f"Total species   : {image_df['species'].nunique()}")
print(f"Total categories: {image_df['category'].nunique()}")
print(f"Total health    : {image_df['health'].nunique()}")

print("\nColumns:")
print(image_df.columns.tolist())

print("\nFirst 5 records:")
display(image_df.head())

In [ ]:
# ============================================================
# CELL 23: EXACT DUPLICATE IMAGE CHECK
# ============================================================

import hashlib
import os
import pandas as pd


def calculate_md5(file_path):
    """Calculate MD5 hash of an image file."""
    
    hash_md5 = hashlib.md5()

    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            hash_md5.update(chunk)

    return hash_md5.hexdigest()


# Calculate hash for every image
image_df["image_hash"] = image_df["image_path"].apply(calculate_md5)


# ------------------------------------------------------------
# Find duplicate images
# ------------------------------------------------------------

duplicate_mask = image_df["image_hash"].duplicated(keep=False)

duplicate_images = image_df[duplicate_mask].copy()


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("EXACT DUPLICATE IMAGE CHECK")
print("=" * 60)

print(f"Total images checked : {len(image_df)}")
print(f"Unique image hashes  : {image_df['image_hash'].nunique()}")
print(f"Duplicate image rows : {duplicate_mask.sum()}")
print(f"Duplicate groups     : "
      f"{duplicate_images['image_hash'].nunique()}")


if len(duplicate_images) == 0:

    print("\n✓ No exact duplicate images detected!")

else:

    print("\n⚠ Duplicate images detected!")

    print("\nDuplicate groups:")
    
    display(
        duplicate_images[
            [
                "image_path",
                "species",
                "category",
                "health",
                "image_hash"
            ]
        ].sort_values("image_hash")
    )

In [ ]:
# ============================================================
# CELL 24: DUPLICATE AND LABEL-CONFLICT ANALYSIS
# ============================================================

# Group images by their exact hash
duplicate_analysis = (
    image_df
    .groupby("image_hash")
    .agg(
        occurrences=("image_path", "count"),
        species_count=("species", "nunique"),
        category_count=("category", "nunique"),
        health_count=("health", "nunique"),
        species=("species", lambda x: " | ".join(sorted(x.unique()))),
        category=("category", lambda x: " | ".join(sorted(x.unique()))),
        health=("health", lambda x: " | ".join(sorted(x.unique())))
    )
    .reset_index()
)

# Keep only actual duplicates
duplicate_analysis = duplicate_analysis[
    duplicate_analysis["occurrences"] > 1
].copy()


# ------------------------------------------------------------
# Categorize duplicate groups
# ------------------------------------------------------------

duplicate_analysis["conflict_type"] = "Same label"

duplicate_analysis.loc[
    duplicate_analysis["health_count"] > 1,
    "conflict_type"
] = "HEALTH LABEL CONFLICT"

duplicate_analysis.loc[
    duplicate_analysis["species_count"] > 1,
    "conflict_type"
] = "SPECIES LABEL CONFLICT"

duplicate_analysis.loc[
    (duplicate_analysis["health_count"] > 1) &
    (duplicate_analysis["species_count"] > 1),
    "conflict_type"
] = "MULTIPLE LABEL CONFLICT"


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("DUPLICATE & LABEL-CONFLICT ANALYSIS")
print("=" * 60)

print(f"Duplicate groups              : {len(duplicate_analysis)}")
print(
    f"Same-label duplicate groups   : "
    f"{(duplicate_analysis['conflict_type'] == 'Same label').sum()}"
)
print(
    f"Health label conflicts        : "
    f"{(duplicate_analysis['conflict_type'] == 'HEALTH LABEL CONFLICT').sum()}"
)
print(
    f"Species label conflicts       : "
    f"{(duplicate_analysis['conflict_type'] == 'SPECIES LABEL CONFLICT').sum()}"
)
print(
    f"Multiple-label conflicts      : "
    f"{(duplicate_analysis['conflict_type'] == 'MULTIPLE LABEL CONFLICT').sum()}"
)


# ------------------------------------------------------------
# Show label conflicts
# ------------------------------------------------------------

label_conflicts = duplicate_analysis[
    duplicate_analysis["conflict_type"] != "Same label"
].copy()

print("\nLABEL CONFLICTS")
print("=" * 60)

if len(label_conflicts) > 0:
    display(
        label_conflicts[
            [
                "image_hash",
                "occurrences",
                "species",
                "category",
                "health",
                "conflict_type"
            ]
        ].sort_values("conflict_type")
    )
else:
    print("✓ No label conflicts detected.")

In [ ]:
# ============================================================
# CELL 25: LOCATE ALL LABEL-CONFLICTING IMAGES
# ============================================================

# Get hashes involved in label conflicts
conflict_hashes = label_conflicts["image_hash"].tolist()

# Extract every image record belonging to those hashes
conflict_images = image_df[
    image_df["image_hash"].isin(conflict_hashes)
].copy()

# Add filename for easier inspection
conflict_images["filename"] = conflict_images["image_path"].apply(
    os.path.basename
)

# Select useful columns
conflict_images = conflict_images[
    [
        "filename",
        "image_path",
        "species",
        "category",
        "health",
        "source",
        "folder_name",
        "image_hash"
    ]
].sort_values(
    ["species", "image_hash", "folder_name"]
).reset_index(drop=True)


print("LABEL-CONFLICTING IMAGE RECORDS")
print("=" * 70)

print(f"Conflict groups : {len(conflict_hashes)}")
print(f"Image records   : {len(conflict_images)}")

display(conflict_images)

In [ ]:
# ============================================================
# CONFLICTING DUPLICATE IMAGES: FILE PATH INSPECTION
# ============================================================

import pandas as pd

# Find duplicate hashes
hash_counts = image_df["image_hash"].value_counts()
duplicate_hashes = hash_counts[hash_counts > 1].index

duplicate_df = image_df[
    image_df["image_hash"].isin(duplicate_hashes)
].copy()

# ------------------------------------------------------------
# 1. Health-label conflicts
# ------------------------------------------------------------

health_conflicts = (
    duplicate_df.groupby("image_hash")["health"]
    .nunique()
)

health_conflict_hashes = health_conflicts[
    health_conflicts > 1
].index

health_conflict_images = duplicate_df[
    duplicate_df["image_hash"].isin(health_conflict_hashes)
].sort_values("image_hash")

print("=" * 70)
print("HEALTH LABEL CONFLICTS")
print("=" * 70)

if len(health_conflict_images) == 0:
    print("No health-label conflicts found.")
else:
    display(
        health_conflict_images[
            ["image_path", "species", "category", "health", "image_hash"]
        ]
    )


# ------------------------------------------------------------
# 2. Species-label conflicts
# ------------------------------------------------------------

species_conflicts = (
    duplicate_df.groupby("image_hash")["species"]
    .nunique()
)

species_conflict_hashes = species_conflicts[
    species_conflicts > 1
].index

species_conflict_images = duplicate_df[
    duplicate_df["image_hash"].isin(species_conflict_hashes)
].sort_values("image_hash")

print("\n" + "=" * 70)
print("SPECIES LABEL CONFLICTS")
print("=" * 70)

if len(species_conflict_images) == 0:
    print("No species-label conflicts found.")
else:
    display(
        species_conflict_images[
            ["image_path", "species", "category", "health", "image_hash"]
        ]
    )


# ------------------------------------------------------------
# 3. Summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CONFLICT SUMMARY")
print("=" * 70)

print("Health conflict groups  :", len(health_conflict_hashes))
print("Species conflict groups :", len(species_conflict_hashes))
print("Total conflicting groups:",
      len(set(health_conflict_hashes) | set(species_conflict_hashes)))

In [ ]:
# ============================================================
# VISUAL INSPECTION OF LABEL-CONFLICTING DUPLICATES
# ============================================================

import os
import math
import matplotlib.pyplot as plt
from PIL import Image

# Combine all conflicting hashes
conflict_hashes = list(
    set(health_conflict_hashes) |
    set(species_conflict_hashes)
)

conflict_images = duplicate_df[
    duplicate_df["image_hash"].isin(conflict_hashes)
].copy()

print("=" * 70)
print("CONFLICTING IMAGES FOR VISUAL INSPECTION")
print("=" * 70)
print(f"Conflict groups : {len(conflict_hashes)}")
print(f"Image rows      : {len(conflict_images)}")


# ------------------------------------------------------------
# Show each conflicting image pair/group
# ------------------------------------------------------------

for image_hash in conflict_hashes:

    group = conflict_images[
        conflict_images["image_hash"] == image_hash
    ]

    print("\n" + "-" * 70)
    print("HASH:", image_hash)

    for idx, row in group.iterrows():
        print(
            f"Row {idx} | "
            f"Species: {row['species']} | "
            f"Health: {row['health']} | "
            f"Path: {row['image_path']}"
        )

    # Display images
    n = len(group)
    fig, axes = plt.subplots(
        1, n,
        figsize=(5 * n, 5)
    )

    if n == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, group.iterrows()):

        try:
            img = Image.open(row["image_path"]).convert("RGB")
            ax.imshow(img)

            ax.set_title(
                f"Species: {row['species']}\n"
                f"Health: {row['health']}\n"
                f"Row: {row.name}",
                fontsize=10
            )

            ax.axis("off")

        except Exception as e:
            ax.set_title(f"ERROR\n{e}")
            ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# STEP: CLEAN LABEL-CONFLICTING DUPLICATES
# ============================================================

import pandas as pd

# image_df contains image-level records + image_hash
clean_df = image_df.copy()

before_count = len(clean_df)

print("=" * 70)
print("CLEANING LABEL-CONFLICTING DUPLICATES")
print("=" * 70)


# ------------------------------------------------------------
# 1. Almond ↔ Chrysanthemum conflicts
# ------------------------------------------------------------

species_conflict_hashes = (
    clean_df.groupby("image_hash")["species"]
    .nunique()
)

species_conflict_hashes = species_conflict_hashes[
    species_conflict_hashes > 1
].index

# Based on visual inspection:
# conflicting images belong to Chrysanthemum,
# so remove the Almond copies.

almond_conflict_rows = clean_df[
    (clean_df["image_hash"].isin(species_conflict_hashes)) &
    (clean_df["species"] == "Almond")
].index

print(f"Almond conflict rows removed : {len(almond_conflict_rows)}")

clean_df = clean_df.drop(index=almond_conflict_rows)


# ------------------------------------------------------------
# 2. Puishak Healthy ↔ Diseased conflicts
# ------------------------------------------------------------

health_conflict_hashes = (
    clean_df.groupby("image_hash")["health"]
    .nunique()
)

health_conflict_hashes = health_conflict_hashes[
    health_conflict_hashes > 1
].index

# Keep Diseased copy and remove Healthy duplicate.

puishak_healthy_conflict_rows = clean_df[
    (clean_df["image_hash"].isin(health_conflict_hashes)) &
    (clean_df["species"] == "Puishak") &
    (clean_df["health"] == "Healthy")
].index

print(
    f"Puishak Healthy conflict rows removed : "
    f"{len(puishak_healthy_conflict_rows)}"
)

clean_df = clean_df.drop(index=puishak_healthy_conflict_rows)


# ------------------------------------------------------------
# Reset index
# ------------------------------------------------------------

clean_df = clean_df.reset_index(drop=True)

after_count = len(clean_df)


# ------------------------------------------------------------
# Cleaning summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLEANING SUMMARY")
print("=" * 70)

print(f"Original images : {before_count}")
print(f"Removed images  : {before_count - after_count}")
print(f"Cleaned images  : {after_count}")


# ------------------------------------------------------------
# Verify remaining conflicts
# ------------------------------------------------------------

print("\nRemaining health conflicts:")

remaining_health_conflicts = (
    clean_df.groupby("image_hash")["health"].nunique()
)

print((remaining_health_conflicts > 1).sum())


print("\nRemaining species conflicts:")

remaining_species_conflicts = (
    clean_df.groupby("image_hash")["species"].nunique()
)

print((remaining_species_conflicts > 1).sum())

In [ ]:
# Check available DataFrame variables

import pandas as pd

df_names = [
    name for name, obj in globals().items()
    if isinstance(obj, pd.DataFrame)
]

print("Available DataFrames:")
print("=" * 50)

for name in df_names:
    print(f"{name:25s} -> {obj.shape if False else globals()[name].shape}")


In [ ]:
print("quality_df columns:")
print(quality_df.columns.tolist())

print("\nShape:", quality_df.shape)

In [ ]:
print("image_df columns:")
print(image_df.columns.tolist())

print("\nShape:", image_df.shape)

In [ ]:
# ============================================================
# CELL 26: REMAINING EXACT DUPLICATE ANALYSIS
# ============================================================

print("=" * 70)
print("REMAINING EXACT DUPLICATE ANALYSIS")
print("=" * 70)

# Count occurrences of each image hash
hash_counts = clean_df["image_hash"].value_counts()

# Keep only hashes appearing more than once
duplicate_hashes = hash_counts[hash_counts > 1]

print(f"Total cleaned images : {len(clean_df)}")
print(f"Unique image hashes  : {clean_df['image_hash'].nunique()}")
print(f"Duplicate groups     : {len(duplicate_hashes)}")
print(f"Duplicate rows       : {duplicate_hashes.sum()}")
print(
    f"Extra duplicate rows : "
    f"{duplicate_hashes.sum() - len(duplicate_hashes)}"
)

print("\nTop remaining duplicate groups:")
print(duplicate_hashes.head(20))

In [ ]:
# ============================================================
# CELL 27: DUPLICATE CONSISTENCY CHECK
# ============================================================

print("=" * 70)
print("DUPLICATE LABEL CONSISTENCY CHECK")
print("=" * 70)

duplicate_hash_list = duplicate_hashes.index

duplicate_records = clean_df[
    clean_df["image_hash"].isin(duplicate_hash_list)
].copy()

# Check species consistency
species_per_hash = (
    duplicate_records
    .groupby("image_hash")["species"]
    .nunique()
)

# Check health consistency
health_per_hash = (
    duplicate_records
    .groupby("image_hash")["health"]
    .nunique()
)

# Check category consistency
category_per_hash = (
    duplicate_records
    .groupby("image_hash")["category"]
    .nunique()
)

print(f"Duplicate groups checked : {len(duplicate_hash_list)}")

print("\nSpecies conflicts among duplicates:")
print((species_per_hash > 1).sum())

print("\nHealth conflicts among duplicates:")
print((health_per_hash > 1).sum())

print("\nCategory conflicts among duplicates:")
print((category_per_hash > 1).sum())


# ------------------------------------------------------------
# Show any remaining conflicts
# ------------------------------------------------------------

remaining_conflict_hashes = set(
    species_per_hash[species_per_hash > 1].index
) | set(
    health_per_hash[health_per_hash > 1].index
) | set(
    category_per_hash[category_per_hash > 1].index
)

print("\n" + "=" * 70)
print("REMAINING DUPLICATE CONFLICTS")
print("=" * 70)

print(
    f"Conflicting duplicate groups : "
    f"{len(remaining_conflict_hashes)}"
)

if remaining_conflict_hashes:
    print("\nConflict records:")
    print(
        duplicate_records[
            duplicate_records["image_hash"].isin(
                remaining_conflict_hashes
            )
        ][
            [
                "image_path",
                "species",
                "category",
                "health",
                "image_hash"
            ]
        ].to_string(index=False)
    )
else:
    print("✓ All remaining duplicates have consistent labels.")

In [ ]:
# ============================================================
# CELL 28: REMOVE EXACT DUPLICATE COPIES
# ============================================================

print("=" * 70)
print("REMOVING EXACT DUPLICATE COPIES")
print("=" * 70)

before_count = len(clean_df)

# Keep only one copy of each exact image
clean_df = clean_df.drop_duplicates(
    subset="image_hash",
    keep="first"
).reset_index(drop=True)

after_count = len(clean_df)

print(f"Images before duplicate removal : {before_count}")
print(f"Duplicate copies removed        : {before_count - after_count}")
print(f"Final unique images             : {after_count}")

print("\nUnique image hashes:")
print(clean_df["image_hash"].nunique())

print("\nRemaining exact duplicate groups:")

remaining_duplicates = (
    clean_df["image_hash"].value_counts()
)

remaining_duplicates = remaining_duplicates[
    remaining_duplicates > 1
]

print(len(remaining_duplicates))

if len(remaining_duplicates) == 0:
    print("✓ No exact duplicate images remain.")

In [ ]:
# ============================================================
# CELL 23: FINAL CLEAN DATASET DISTRIBUTION
# ============================================================

print("=" * 70)
print("FINAL CLEAN DATASET DISTRIBUTION")
print("=" * 70)

print(f"Total images    : {len(clean_df)}")
print(f"Total species   : {clean_df['species'].nunique()}")
print(f"Total categories: {clean_df['category'].nunique()}")
print(f"Total health    : {clean_df['health'].nunique()}")

# ------------------------------------------------------------
# 1. Species distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SPECIES-WISE DISTRIBUTION")
print("=" * 70)

species_dist = (
    clean_df["species"]
    .value_counts()
    .sort_values(ascending=False)
)

display(species_dist.to_frame("image_count"))


# ------------------------------------------------------------
# 2. Health distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("HEALTH-WISE DISTRIBUTION")
print("=" * 70)

health_dist = (
    clean_df["health"]
    .value_counts()
)

display(health_dist.to_frame("image_count"))


# ------------------------------------------------------------
# 3. Species × Health distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SPECIES × HEALTH DISTRIBUTION")
print("=" * 70)

species_health_dist = pd.crosstab(
    clean_df["species"],
    clean_df["health"]
)

display(species_health_dist)


# ------------------------------------------------------------
# 4. Category distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CATEGORY DISTRIBUTION")
print("=" * 70)

category_dist = (
    clean_df["category"]
    .value_counts()
)

display(category_dist.to_frame("image_count"))


# ------------------------------------------------------------
# 5. Smallest classes
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SMALLEST SPECIES CLASSES")
print("=" * 70)

display(
    species_dist
    .sort_values()
    .head(10)
    .to_frame("image_count")
)


# ------------------------------------------------------------
# 6. Largest classes
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LARGEST SPECIES CLASSES")
print("=" * 70)

display(
    species_dist
    .head(10)
    .to_frame("image_count")
)

print("\n" + "=" * 70)
print("✓ FINAL DATASET CHECK COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 24: FINAL 70/15/15 STRATIFIED SPLIT
# ============================================================

from sklearn.model_selection import train_test_split
import pandas as pd

print("=" * 70)
print("CREATING FINAL STRATIFIED 70/15/15 SPLIT")
print("=" * 70)

df = clean_df.copy()

# ------------------------------------------------------------
# Combined species + health label
# ------------------------------------------------------------

df["stratify_label"] = (
    df["species"].astype(str)
    + "__"
    + df["health"].astype(str)
)

# ============================================================
# STEP 1: 70% TRAIN / 30% TEMP
# ============================================================

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["stratify_label"]
)

print(f"\nTrain      : {len(train_df)}")
print(f"Temporary  : {len(temp_df)}")


# ============================================================
# STEP 2: TEMP → VALIDATION + TEST
#
# Some tiny groups have only 1 sample inside temp_df.
# Therefore, use species-level stratification where possible.
# For species that still have only 1 sample, don't stratify them.
# ============================================================

temp_species_counts = temp_df["species"].value_counts()

# Species having at least 2 samples can participate in stratification
valid_species = temp_species_counts[
    temp_species_counts >= 2
].index

stratified_temp = temp_df[
    temp_df["species"].isin(valid_species)
].copy()

unstratified_temp = temp_df[
    ~temp_df["species"].isin(valid_species)
].copy()

print("\nTemporary set:")
print(f"Stratified portion   : {len(stratified_temp)}")
print(f"Unstratified portion : {len(unstratified_temp)}")


# ------------------------------------------------------------
# Split the stratifiable portion
# ------------------------------------------------------------

val_part, test_part = train_test_split(
    stratified_temp,
    test_size=0.50,
    random_state=42,
    stratify=stratified_temp["species"]
)


# ------------------------------------------------------------
# Split the tiny/unstratifiable samples manually
# ------------------------------------------------------------

# Deterministic split so results remain reproducible
unstratified_temp = unstratified_temp.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

n_test_extra = len(unstratified_temp) // 2

test_extra = unstratified_temp.iloc[:n_test_extra]
val_extra = unstratified_temp.iloc[n_test_extra:]


# ------------------------------------------------------------
# Combine
# ------------------------------------------------------------

val_df = pd.concat(
    [val_part, val_extra],
    ignore_index=True
)

test_df = pd.concat(
    [test_part, test_extra],
    ignore_index=True
)


# ============================================================
# RESET INDEX
# ============================================================

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)


# Remove helper column
train_df = train_df.drop(columns=["stratify_label"])
val_df = val_df.drop(columns=["stratify_label"])
test_df = test_df.drop(columns=["stratify_label"])


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FINAL SPLIT SUMMARY")
print("=" * 70)

total = len(df)

print(f"Total images      : {total}")
print(f"Training images   : {len(train_df)} ({len(train_df)/total*100:.2f}%)")
print(f"Validation images : {len(val_df)} ({len(val_df)/total*100:.2f}%)")
print(f"Test images       : {len(test_df)} ({len(test_df)/total*100:.2f}%)")


# ============================================================
# SPECIES DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("SPECIES DISTRIBUTION")
print("=" * 70)

species_split = pd.DataFrame({
    "Total": df["species"].value_counts(),
    "Train": train_df["species"].value_counts(),
    "Validation": val_df["species"].value_counts(),
    "Test": test_df["species"].value_counts()
}).fillna(0).astype(int)

display(species_split)


# ============================================================
# HEALTH DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("HEALTH DISTRIBUTION")
print("=" * 70)

health_split = pd.DataFrame({
    "Total": df["health"].value_counts(),
    "Train": train_df["health"].value_counts(),
    "Validation": val_df["health"].value_counts(),
    "Test": test_df["health"].value_counts()
}).fillna(0).astype(int)

display(health_split)


# ============================================================
# CATEGORY DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("CATEGORY DISTRIBUTION")
print("=" * 70)

category_split = pd.DataFrame({
    "Total": df["category"].value_counts(),
    "Train": train_df["category"].value_counts(),
    "Validation": val_df["category"].value_counts(),
    "Test": test_df["category"].value_counts()
}).fillna(0).astype(int)

display(category_split)


print("\n" + "=" * 70)
print("✓ FINAL 70/15/15 SPLIT COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 25: DATA LEAKAGE CHECK
# ============================================================

print("=" * 70)
print("DATA LEAKAGE CHECK")
print("=" * 70)

# ------------------------------------------------------------
# 1. Image path overlap
# ------------------------------------------------------------

train_paths = set(train_df["image_path"])
val_paths   = set(val_df["image_path"])
test_paths  = set(test_df["image_path"])

train_val_overlap = train_paths & val_paths
train_test_overlap = train_paths & test_paths
val_test_overlap = val_paths & test_paths

print("\nIMAGE PATH OVERLAP")
print("-" * 70)

print(f"Train ∩ Validation : {len(train_val_overlap)}")
print(f"Train ∩ Test       : {len(train_test_overlap)}")
print(f"Validation ∩ Test  : {len(val_test_overlap)}")


# ------------------------------------------------------------
# 2. Exact image hash overlap
# ------------------------------------------------------------

# image_hash may not be present if it was removed earlier.
# Recover it from the original image_df if necessary.

if "image_hash" not in train_df.columns:

    hash_lookup = image_df[
        ["image_path", "image_hash"]
    ].drop_duplicates("image_path")

    train_check = train_df.merge(
        hash_lookup,
        on="image_path",
        how="left"
    )

    val_check = val_df.merge(
        hash_lookup,
        on="image_path",
        how="left"
    )

    test_check = test_df.merge(
        hash_lookup,
        on="image_path",
        how="left"
    )

else:

    train_check = train_df.copy()
    val_check = val_df.copy()
    test_check = test_df.copy()


train_hashes = set(
    train_check["image_hash"].dropna()
)

val_hashes = set(
    val_check["image_hash"].dropna()
)

test_hashes = set(
    test_check["image_hash"].dropna()
)

train_val_hash_overlap = train_hashes & val_hashes
train_test_hash_overlap = train_hashes & test_hashes
val_test_hash_overlap = val_hashes & test_hashes


print("\nEXACT IMAGE HASH OVERLAP")
print("-" * 70)

print(f"Train ∩ Validation : {len(train_val_hash_overlap)}")
print(f"Train ∩ Test       : {len(train_test_hash_overlap)}")
print(f"Validation ∩ Test  : {len(val_test_hash_overlap)}")


# ------------------------------------------------------------
# 3. Label consistency
# ------------------------------------------------------------

print("\nLABEL CONSISTENCY")
print("-" * 70)

print(
    "Train species labels   :",
    train_df["species"].nunique()
)

print(
    "Validation species     :",
    val_df["species"].nunique()
)

print(
    "Test species           :",
    test_df["species"].nunique()
)


# ------------------------------------------------------------
# 4. Final verdict
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LEAKAGE CHECK RESULT")
print("=" * 70)

total_overlap = (
    len(train_val_overlap)
    + len(train_test_overlap)
    + len(val_test_overlap)
    + len(train_val_hash_overlap)
    + len(train_test_hash_overlap)
    + len(val_test_hash_overlap)
)

if total_overlap == 0:

    print("✓ No image-path overlap detected.")
    print("✓ No exact image-hash overlap detected.")
    print("✓ Train / Validation / Test are leakage-free.")

else:

    print("⚠ Potential data leakage detected!")
    print("Please inspect the overlap results above.")

print("=" * 70)

In [ ]:
# ============================================================
# CELL 22: SPECIES LABEL ENCODING
# ============================================================

from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np

print("=" * 70)
print("SPECIES LABEL ENCODING")
print("=" * 70)

# ------------------------------------------------------------
# Create label encoder
# ------------------------------------------------------------

label_encoder = LabelEncoder()

# Fit ONLY on the complete cleaned dataset's species names
# This guarantees one consistent mapping across all splits.
label_encoder.fit(clean_df["species"])

# ------------------------------------------------------------
# Encode labels
# ------------------------------------------------------------

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["label"] = label_encoder.transform(train_df["species"])
val_df["label"] = label_encoder.transform(val_df["species"])
test_df["label"] = label_encoder.transform(test_df["species"])

# ------------------------------------------------------------
# Number of classes
# ------------------------------------------------------------

num_classes = len(label_encoder.classes_)

print(f"Number of species/classes : {num_classes}")

print("\nSpecies → Label mapping:")
print("-" * 70)

for idx, species in enumerate(label_encoder.classes_):
    print(f"{idx:2d} -> {species}")

# ------------------------------------------------------------
# Verify labels
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LABEL DISTRIBUTION")
print("=" * 70)

print(f"Train labels      : {train_df['label'].nunique()}")
print(f"Validation labels : {val_df['label'].nunique()}")
print(f"Test labels       : {test_df['label'].nunique()}")

print("\nLabel encoding complete ✓")

In [ ]:
# ============================================================
# CELL 23: IMAGE PREPROCESSING + AUGMENTATION
# ============================================================

import torch
from torchvision import transforms

print("=" * 70)
print("IMAGE PREPROCESSING & AUGMENTATION")
print("=" * 70)

# ------------------------------------------------------------
# Image configuration
# ------------------------------------------------------------

IMG_SIZE = 224

print(f"Image size : {IMG_SIZE} x {IMG_SIZE}")

# ------------------------------------------------------------
# Training transformations
# ------------------------------------------------------------
# Augmentation is applied ONLY to training images.

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),

    transforms.RandomRotation(
        degrees=20
    ),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),

    transforms.RandomResizedCrop(
        IMG_SIZE,
        scale=(0.80, 1.0),
        ratio=(0.90, 1.10)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ------------------------------------------------------------
# Validation transformations
# ------------------------------------------------------------
# NO random augmentation.

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ------------------------------------------------------------
# Test transformations
# ------------------------------------------------------------
# Test set must remain completely deterministic.

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nTraining augmentation:")
print("  ✓ Horizontal flip")
print("  ✓ Vertical flip")
print("  ✓ Rotation")
print("  ✓ Color jitter")
print("  ✓ Random resized crop")

print("\nValidation:")
print("  ✓ Resize")
print("  ✓ Normalize")
print("  ✗ Random augmentation")

print("\nTest:")
print("  ✓ Resize")
print("  ✓ Normalize")
print("  ✗ Random augmentation")

print("\n" + "=" * 70)
print("TRANSFORMS READY ✓")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 24: PYTORCH DATASET
# ============================================================

from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch
import os

print("=" * 70)
print("CREATING PYTORCH DATASET")
print("=" * 70)


# ------------------------------------------------------------
# Custom Dataset class
# ------------------------------------------------------------

class LeafDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        image_path = row["image_path"]
        label = int(row["label"])

        # Load image
        image = Image.open(image_path).convert("RGB")

        # Apply transformation
        if self.transform is not None:
            image = self.transform(image)

        return image, label


# ------------------------------------------------------------
# Create datasets
# ------------------------------------------------------------

train_dataset = LeafDataset(
    train_df,
    transform=train_transform
)

val_dataset = LeafDataset(
    val_df,
    transform=val_transform
)

test_dataset = LeafDataset(
    test_df,
    transform=test_transform
)


# ------------------------------------------------------------
# Dataset sizes
# ------------------------------------------------------------

print(f"Training dataset   : {len(train_dataset)}")
print(f"Validation dataset : {len(val_dataset)}")
print(f"Test dataset       : {len(test_dataset)}")


# ------------------------------------------------------------
# DataLoader configuration
# ------------------------------------------------------------

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ------------------------------------------------------------
# Verify one batch
# ------------------------------------------------------------

images, labels = next(iter(train_loader))

print("\n" + "=" * 70)
print("BATCH VERIFICATION")
print("=" * 70)

print(f"Image batch shape : {images.shape}")
print(f"Label batch shape : {labels.shape}")
print(f"Image dtype       : {images.dtype}")
print(f"Label dtype       : {labels.dtype}")

print("\nExpected:")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  Channels   : 3")
print(f"  Height     : {IMG_SIZE}")
print(f"  Width      : {IMG_SIZE}")

print("\n" + "=" * 70)
print("PYTORCH DATASET & DATALOADERS READY ✓")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 25: CLASS / LABEL MAPPING VERIFICATION
# ============================================================

print("=" * 70)
print("CLASS / LABEL MAPPING VERIFICATION")
print("=" * 70)

# Create class names from the final cleaned dataset
class_names = sorted(image_df["species"].unique().tolist())

# Create class-to-index mapping
class_to_idx = {
    name: idx
    for idx, name in enumerate(class_names)
}

idx_to_class = {
    idx: name
    for name, idx in class_to_idx.items()
}

print(f"Total classes : {len(class_names)}")

print("\nClass index mapping:")
print("-" * 70)

for idx, name in enumerate(class_names):
    print(f"{idx:2d} -> {name}")

print("\n" + "=" * 70)
print("LABEL RANGE CHECK")
print("=" * 70)

print(f"Minimum label in batch : {labels.min().item()}")
print(f"Maximum label in batch : {labels.max().item()}")

assert labels.min().item() >= 0
assert labels.max().item() < len(class_names)

print("\n✓ All labels are within valid class range.")
print("✓ Class mapping verified.")

print("=" * 70)

In [ ]:
# ============================================================
# CELL 26: VERIFY DATASET LABEL ENCODING
# ============================================================

print("=" * 70)
print("DATASET LABEL ENCODING VERIFICATION")
print("=" * 70)

print(f"Number of classes : {len(class_names)}")
print(f"Number of mappings: {len(class_to_idx)}")

# Check that every class has exactly one index
assert len(class_names) == 33
assert len(class_to_idx) == 33
assert set(class_names) == set(class_to_idx.keys())

print("\nClass-to-index mapping:")
print("-" * 70)

for class_name in class_names:
    print(f"{class_to_idx[class_name]:2d} -> {class_name}")

print("\n" + "=" * 70)
print("✓ Dataset label encoding verified.")
print("✓ 33 classes are mapped uniquely.")
print("=" * 70)

In [ ]:
# Optional Kaggle-only recovery command from the original development session (not run automatically).

In [ ]:
import os

output_dir = os.path.join(OUTPUT_ROOT, "old_outputs")

print("=== Available model checkpoints ===")

for root, dirs, files in os.walk(output_dir):
    for file in sorted(files):
        if file.endswith((".pth", ".pt", ".ckpt")):
            print(file)

In [ ]:
# Optional Kaggle-only recovery command from the original development session (not run automatically).

In [ ]:
import os

for root, dirs, files in os.walk(os.path.join(OUTPUT_ROOT, "old_outputs")):
    for file in files:
        print(os.path.join(root, file))

In [ ]:

import json
import os

class_names_path = os.path.join(OUTPUT_ROOT, "old_outputs", "class_names.json")

with open(class_names_path, "r") as f:
    class_names = json.load(f)

print("Number of classes:", len(class_names))
print("Class names:", class_names)


In [ ]:
# ============================================================
# CELL 95: PURE VISION TRANSFORMER (ViT-B/16) SETUP
# ============================================================

import torch
import torch.nn as nn
from torchvision import models

print("=" * 80)
print("PURE VISION TRANSFORMER (ViT-B/16) SETUP")
print("=" * 80)

# ------------------------------------------------------------
# 1. Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Device: {device}")

# ------------------------------------------------------------
# 2. Number of Classes
# ------------------------------------------------------------

NUM_CLASSES = len(class_names)

print(f"Number of Classes: {NUM_CLASSES}")

# ------------------------------------------------------------
# 3. Load ImageNet Pretrained ViT-B/16
# ------------------------------------------------------------

vit_weights = models.ViT_B_16_Weights.DEFAULT

vit_model = models.vit_b_16(
    weights=vit_weights
)

# ------------------------------------------------------------
# 4. Freeze ViT Backbone
# ------------------------------------------------------------

for param in vit_model.parameters():
    param.requires_grad = False

# ------------------------------------------------------------
# 5. Replace Classification Head
# ------------------------------------------------------------

in_features = vit_model.heads.head.in_features

vit_model.heads.head = nn.Linear(
    in_features,
    NUM_CLASSES
)

# Make only the new classification head trainable
for param in vit_model.heads.head.parameters():
    param.requires_grad = True

# ------------------------------------------------------------
# 6. Move Model to Device
# ------------------------------------------------------------

vit_model = vit_model.to(device)

# ------------------------------------------------------------
# 7. Parameter Count
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in vit_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in vit_model.parameters()
    if p.requires_grad
)

frozen_params = total_params - trainable_params

# ------------------------------------------------------------
# 8. Model Summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MODEL ARCHITECTURE")
print("=" * 80)

print(vit_model)

print("\n" + "=" * 80)
print("PARAMETER SUMMARY")
print("=" * 80)

print(f"Total Parameters      : {total_params:,}")
print(f"Trainable Parameters  : {trainable_params:,}")
print(f"Frozen Parameters     : {frozen_params:,}")

print("\n" + "=" * 80)
print("CLASSIFIER INFORMATION")
print("=" * 80)

print(f"Original Head Input   : {in_features}")
print(f"Output Classes        : {NUM_CLASSES}")
print("Architecture          : ViT-B/16 → Classification Head")
print("Pretrained            : ImageNet")
print("Backbone Frozen       : Yes")

print("\n" + "=" * 80)
print("✓ PURE VIT SETUP COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 96: PURE ViT-B/16 4-FOLD CROSS-VALIDATION TRAINING
# ============================================================

import torch
import torch.nn as nn
import numpy as np
import copy
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

print("=" * 80)
print("PURE ViT-B/16 4-FOLD CROSS-VALIDATION TRAINING")
print("=" * 80)

# ------------------------------------------------------------
# 1. Training Configuration
# ------------------------------------------------------------

NUM_FOLDS = 4
MAX_EPOCHS = 150
EARLY_STOPPING_PATIENCE = 10

INITIAL_LR = 0.001
WEIGHT_DECAY = 0.0001
BATCH_SIZE = 32

print("\nTraining Configuration:")
print(f"Folds                  : {NUM_FOLDS}")
print(f"Maximum Epochs         : {MAX_EPOCHS}")
print(f"Early Stopping         : {EARLY_STOPPING_PATIENCE}")
print(f"Learning Rate          : {INITIAL_LR}")
print(f"Weight Decay           : {WEIGHT_DECAY}")
print(f"Batch Size             : {BATCH_SIZE}")

# ------------------------------------------------------------
# 2. Extract Labels from Training Dataset
# ------------------------------------------------------------

all_dataset_labels = []

for i in range(len(train_dataset)):

    label = train_dataset[i][1]

    if torch.is_tensor(label):
        label = label.item()

    all_dataset_labels.append(int(label))

all_dataset_labels = np.array(
    all_dataset_labels
)

print(f"\nTraining Samples: {len(all_dataset_labels)}")
print(f"Number of Classes: {len(np.unique(all_dataset_labels))}")

# ------------------------------------------------------------
# 3. Stratified K-Fold
# ------------------------------------------------------------

skf = StratifiedKFold(
    n_splits=NUM_FOLDS,
    shuffle=True,
    random_state=42
)

vit_fold_results = []

global_best_acc = 0.0
global_best_state = None
global_best_fold = None
global_best_epoch = None

# ------------------------------------------------------------
# 4. Cross-Validation
# ------------------------------------------------------------

for fold, (train_idx, val_idx) in enumerate(
    skf.split(
        np.zeros(len(all_dataset_labels)),
        all_dataset_labels
    ),
    start=1
):

    print("\n" + "=" * 80)
    print(f"FOLD {fold}/{NUM_FOLDS}")
    print("=" * 80)

    # --------------------------------------------------------
    # Create fold datasets
    # --------------------------------------------------------

    fold_train_dataset = Subset(
        train_dataset,
        train_idx
    )

    fold_val_dataset = Subset(
        train_dataset,
        val_idx
    )

    fold_train_loader = DataLoader(
        fold_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

    fold_val_loader = DataLoader(
        fold_val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

    # --------------------------------------------------------
    # Fresh ViT model for each fold
    # --------------------------------------------------------

    fold_model = models.vit_b_16(
        weights=vit_weights
    )

    # Freeze backbone
    for param in fold_model.parameters():
        param.requires_grad = False

    # Replace classification head
    fold_model.heads.head = nn.Linear(
        fold_model.heads.head.in_features,
        NUM_CLASSES
    )

    # Train only classifier
    for param in fold_model.heads.head.parameters():
        param.requires_grad = True

    fold_model = fold_model.to(device)

    # --------------------------------------------------------
    # Loss, Optimizer, Scheduler
    # --------------------------------------------------------

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        fold_model.heads.head.parameters(),
        lr=INITIAL_LR,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=3
    )

    # --------------------------------------------------------
    # Tracking
    # --------------------------------------------------------

    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_epoch = 0
    patience_counter = 0

    fold_history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    # --------------------------------------------------------
    # Training Loop
    # --------------------------------------------------------

    for epoch in range(1, MAX_EPOCHS + 1):

        # ====================================================
        # TRAIN
        # ====================================================

        fold_model.train()

        train_running_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in fold_train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = fold_model(images)

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            train_running_loss += (
                loss.item() * images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            train_correct += (
                predictions == labels
            ).sum().item()

            train_total += labels.size(0)

        train_loss = (
            train_running_loss / train_total
        )

        train_acc = (
            train_correct / train_total
        ) * 100

        # ====================================================
        # VALIDATION
        # ====================================================

        fold_model.eval()

        val_running_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for images, labels in fold_val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = fold_model(images)

                loss = criterion(
                    outputs,
                    labels
                )

                val_running_loss += (
                    loss.item() * images.size(0)
                )

                predictions = torch.argmax(
                    outputs,
                    dim=1
                )

                val_correct += (
                    predictions == labels
                ).sum().item()

                val_total += labels.size(0)

        val_loss = (
            val_running_loss / val_total
        )

        val_acc = (
            val_correct / val_total
        ) * 100

        scheduler.step(val_loss)

        # ----------------------------------------------------
        # Store History
        # ----------------------------------------------------

        fold_history["train_loss"].append(train_loss)
        fold_history["train_acc"].append(train_acc)
        fold_history["val_loss"].append(val_loss)
        fold_history["val_acc"].append(val_acc)

        # ----------------------------------------------------
        # Best Model
        # ----------------------------------------------------

        if val_acc > best_val_acc:

            best_val_acc = val_acc
            best_val_loss = val_loss
            best_epoch = epoch

            best_fold_state = copy.deepcopy(
                fold_model.state_dict()
            )

            patience_counter = 0

        else:

            patience_counter += 1

        # ----------------------------------------------------
        # Print Progress
        # ----------------------------------------------------

        if (
            epoch == 1
            or epoch % 5 == 0
            or epoch == best_epoch
        ):

            print(
                f"Epoch {epoch:3d}/{MAX_EPOCHS} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Train Acc: {train_acc:6.2f}% | "
                f"Val Loss: {val_loss:.4f} | "
                f"Val Acc: {val_acc:6.2f}%"
            )

        # ----------------------------------------------------
        # Early Stopping
        # ----------------------------------------------------

        if patience_counter >= EARLY_STOPPING_PATIENCE:

            print(
                f"\nEarly stopping at epoch {epoch}"
            )

            break

    # --------------------------------------------------------
    # Save Best Fold Model
    # --------------------------------------------------------

    fold_model.load_state_dict(
        best_fold_state
    )

    fold_model_path = (
        f"vit_fold_{fold}.pth"
    )

    torch.save(
        best_fold_state,
        fold_model_path
    )

    # --------------------------------------------------------
    # Store Fold Result
    # --------------------------------------------------------

    vit_fold_results.append({
        "fold": fold,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "best_val_acc": best_val_acc,
        "history": fold_history
    })

    # --------------------------------------------------------
    # Global Best
    # --------------------------------------------------------

    if best_val_acc > global_best_acc:

        global_best_acc = best_val_acc
        global_best_state = copy.deepcopy(
            best_fold_state
        )
        global_best_fold = fold
        global_best_epoch = best_epoch

    print(
        f"\nFold {fold} Best Validation Accuracy: "
        f"{best_val_acc:.2f}%"
    )

# ------------------------------------------------------------
# 5. Save Global Best ViT
# ------------------------------------------------------------

torch.save(
    global_best_state,
    "best_vit.pth"
)

# ------------------------------------------------------------
# 6. Cross-Validation Summary
# ------------------------------------------------------------

fold_accuracies = [
    result["best_val_acc"]
    for result in vit_fold_results
]

mean_cv_accuracy = np.mean(
    fold_accuracies
)

std_cv_accuracy = np.std(
    fold_accuracies
)

print("\n" + "=" * 80)
print("PURE ViT CROSS-VALIDATION SUMMARY")
print("=" * 80)

for result in vit_fold_results:

    print(
        f"Fold {result['fold']}: "
        f"Best Epoch = {result['best_epoch']}, "
        f"Best Val Acc = {result['best_val_acc']:.2f}%"
    )

print("\n" + "-" * 80)

print(
    f"Mean CV Accuracy : "
    f"{mean_cv_accuracy:.2f}%"
)

print(
    f"Std CV Accuracy  : "
    f"{std_cv_accuracy:.2f}%"
)

print(
    f"Minimum Accuracy : "
    f"{min(fold_accuracies):.2f}%"
)

print(
    f"Maximum Accuracy : "
    f"{max(fold_accuracies):.2f}%"
)

print("\nGlobal Best Fold:")
print(f"Fold              : {global_best_fold}")
print(f"Best Epoch        : {global_best_epoch}")
print(f"Validation Acc    : {global_best_acc:.2f}%")

print("\n✓ Saved:")
print("  vit_fold_1.pth")
print("  vit_fold_2.pth")
print("  vit_fold_3.pth")
print("  vit_fold_4.pth")
print("  best_vit.pth")

print("=" * 80)
print("✓ PURE ViT CROSS-VALIDATION COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 97: PURE ViT CV RESULTS & VISUALIZATION
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("=" * 80)
print("PURE ViT CROSS-VALIDATION RESULTS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Create Fold-wise Results Table
# ------------------------------------------------------------

vit_cv_table = pd.DataFrame([
    {
        "fold": result["fold"],
        "best_epoch": result["best_epoch"],
        "best_val_loss": result["best_val_loss"],
        "best_val_acc": result["best_val_acc"]
    }
    for result in vit_fold_results
])

print("\nFold-wise Validation Performance:")
print("-" * 80)

print(
    vit_cv_table.to_string(
        index=False,
        formatters={
            "best_val_loss": "{:.4f}".format,
            "best_val_acc": "{:.2f}%".format
        }
    )
)

# ------------------------------------------------------------
# 2. CV Summary
# ------------------------------------------------------------

fold_accuracies = vit_cv_table["best_val_acc"].values

vit_cv_mean = np.mean(fold_accuracies)
vit_cv_std = np.std(fold_accuracies)
vit_cv_min = np.min(fold_accuracies)
vit_cv_max = np.max(fold_accuracies)

best_fold_row = vit_cv_table.loc[
    vit_cv_table["best_val_acc"].idxmax()
]

print("\n" + "=" * 80)
print("CROSS-VALIDATION SUMMARY")
print("=" * 80)

print(
    f"Mean CV Accuracy : {vit_cv_mean:.2f}%"
)

print(
    f"Std CV Accuracy  : {vit_cv_std:.2f}%"
)

print(
    f"Minimum Accuracy : {vit_cv_min:.2f}%"
)

print(
    f"Maximum Accuracy : {vit_cv_max:.2f}%"
)

print("\nBest Fold:")
print(
    f"Fold           : {int(best_fold_row['fold'])}"
)

print(
    f"Best Epoch     : {int(best_fold_row['best_epoch'])}"
)

print(
    f"Validation Acc : {best_fold_row['best_val_acc']:.2f}%"
)

# ------------------------------------------------------------
# 3. Fold Accuracy Visualization
# ------------------------------------------------------------

plt.figure(figsize=(11, 7))

bars = plt.bar(
    vit_cv_table["fold"].astype(str),
    vit_cv_table["best_val_acc"]
)

plt.axhline(
    vit_cv_mean,
    linestyle="--",
    linewidth=2,
    label=f"Mean CV Accuracy = {vit_cv_mean:.2f}%"
)

plt.xlabel(
    "Fold",
    fontsize=12
)

plt.ylabel(
    "Validation Accuracy (%)",
    fontsize=12
)

plt.title(
    "Pure ViT-B/16: 4-Fold Cross-Validation Accuracy",
    fontsize=16,
    fontweight="bold"
)

plt.ylim(0, 100)

for bar, value in zip(
    bars,
    vit_cv_table["best_val_acc"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.2f}%",
        ha="center",
        fontsize=10
    )

plt.legend()
plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "vit_cv_accuracy.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 4. Best Epoch Visualization
# ------------------------------------------------------------

plt.figure(figsize=(11, 7))

bars = plt.bar(
    vit_cv_table["fold"].astype(str),
    vit_cv_table["best_epoch"]
)

plt.xlabel(
    "Fold",
    fontsize=12
)

plt.ylabel(
    "Best Epoch",
    fontsize=12
)

plt.title(
    "Pure ViT-B/16: Best Epoch per Fold",
    fontsize=16,
    fontweight="bold"
)

for bar, value in zip(
    bars,
    vit_cv_table["best_epoch"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{int(value)}",
        ha="center",
        fontsize=10
    )

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "vit_best_epoch_per_fold.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 5. Save Results
# ------------------------------------------------------------

vit_cv_table.to_csv(
    "vit_cv_results.csv",
    index=False
)

print("\n✓ Saved:")
print("  vit_cv_results.csv")
print("  vit_cv_accuracy.png")
print("  vit_best_epoch_per_fold.png")

print("\n" + "=" * 80)
print("✓ PURE ViT CV RESULTS & VISUALIZATION COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 98: PURE ViT TRAINING HISTORY
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

print("=" * 80)
print("PURE ViT-B/16 TRAINING HISTORY")
print("=" * 80)

# ------------------------------------------------------------
# 1. Plot Training & Validation Accuracy
# ------------------------------------------------------------

plt.figure(figsize=(12, 8))

for result in vit_fold_results:

    history = result["history"]

    epochs = np.arange(
        1,
        len(history["train_acc"]) + 1
    )

    plt.plot(
        epochs,
        history["train_acc"],
        linewidth=2,
        label=f"Fold {result['fold']} Train"
    )

    plt.plot(
        epochs,
        history["val_acc"],
        linestyle="--",
        linewidth=2,
        label=f"Fold {result['fold']} Validation"
    )

plt.xlabel(
    "Epoch",
    fontsize=12
)

plt.ylabel(
    "Accuracy (%)",
    fontsize=12
)

plt.title(
    "Pure ViT-B/16: Training vs Validation Accuracy",
    fontsize=16,
    fontweight="bold"
)

plt.legend(
    ncol=2,
    fontsize=9
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "vit_training_validation_accuracy.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 2. Plot Training & Validation Loss
# ------------------------------------------------------------

plt.figure(figsize=(12, 8))

for result in vit_fold_results:

    history = result["history"]

    epochs = np.arange(
        1,
        len(history["train_loss"]) + 1
    )

    plt.plot(
        epochs,
        history["train_loss"],
        linewidth=2,
        label=f"Fold {result['fold']} Train"
    )

    plt.plot(
        epochs,
        history["val_loss"],
        linestyle="--",
        linewidth=2,
        label=f"Fold {result['fold']} Validation"
    )

plt.xlabel(
    "Epoch",
    fontsize=12
)

plt.ylabel(
    "Loss",
    fontsize=12
)

plt.title(
    "Pure ViT-B/16: Training vs Validation Loss",
    fontsize=16,
    fontweight="bold"
)

plt.legend(
    ncol=2,
    fontsize=9
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "vit_training_validation_loss.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 3. Training History Summary
# ------------------------------------------------------------

print("\nTRAINING HISTORY SUMMARY")
print("-" * 80)

for result in vit_fold_results:

    history = result["history"]

    print(
        f"Fold {result['fold']}: "
        f"{len(history['train_acc'])} epochs | "
        f"Best Val Acc = {result['best_val_acc']:.2f}% | "
        f"Best Epoch = {result['best_epoch']}"
    )

# ------------------------------------------------------------
# 4. Save
# ------------------------------------------------------------

print("\n✓ Saved:")
print("  vit_training_validation_accuracy.png")
print("  vit_training_validation_loss.png")

print("=" * 80)
print("✓ PURE ViT TRAINING HISTORY COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 99: PURE ViT-B/16 FINAL TEST EVALUATION
# ============================================================

import torch
import numpy as np
from sklearn.metrics import accuracy_score

print("=" * 80)
print("PURE ViT-B/16 FINAL TEST EVALUATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. Create Fresh ViT Model
# ------------------------------------------------------------

final_vit_model = models.vit_b_16(
    weights=None
)

# Replace classification head
final_vit_model.heads.head = nn.Linear(
    final_vit_model.heads.head.in_features,
    NUM_CLASSES
)

# ------------------------------------------------------------
# 2. Load Best CV Model
# ------------------------------------------------------------

vit_checkpoint_path = "best_vit.pth"

checkpoint = torch.load(
    vit_checkpoint_path,
    map_location=device
)

# Handle state_dict directly or checkpoint dictionary
if (
    isinstance(checkpoint, dict)
    and "model_state_dict" in checkpoint
):

    final_vit_model.load_state_dict(
        checkpoint["model_state_dict"]
    )

else:

    final_vit_model.load_state_dict(
        checkpoint
    )

final_vit_model = final_vit_model.to(device)
final_vit_model.eval()

print(f"✓ Model loaded: {vit_checkpoint_path}")
print(f"✓ Device: {device}")

# ------------------------------------------------------------
# 3. Test Evaluation
# ------------------------------------------------------------

vit_test_true = []
vit_test_pred = []
vit_test_probs = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = final_vit_model(
            images
        )

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        vit_test_true.extend(
            labels.cpu().numpy()
        )

        vit_test_pred.extend(
            predictions.cpu().numpy()
        )

        vit_test_probs.extend(
            probabilities.cpu().numpy()
        )

# ------------------------------------------------------------
# 4. Convert to NumPy
# ------------------------------------------------------------

vit_test_true = np.array(
    vit_test_true
)

vit_test_pred = np.array(
    vit_test_pred
)

vit_test_probs = np.array(
    vit_test_probs
)

# ------------------------------------------------------------
# 5. Calculate Accuracy
# ------------------------------------------------------------

vit_test_accuracy = (
    accuracy_score(
        vit_test_true,
        vit_test_pred
    ) * 100
)

correct_predictions = np.sum(
    vit_test_true == vit_test_pred
)

incorrect_predictions = np.sum(
    vit_test_true != vit_test_pred
)

# ------------------------------------------------------------
# 6. Final Result
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PURE ViT FINAL TEST RESULT")
print("=" * 80)

print(
    f"Test Samples       : {len(vit_test_true)}"
)

print(
    f"Correct Predictions: {correct_predictions}"
)

print(
    f"Incorrect Predictions: {incorrect_predictions}"
)

print(
    f"Test Accuracy      : {vit_test_accuracy:.2f}%"
)

print("=" * 80)
print("✓ PURE ViT TEST EVALUATION COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 100: PURE ViT CLASSIFICATION REPORT
# ============================================================

from sklearn.metrics import (
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

print("=" * 80)
print("PURE ViT-B/16 CLASSIFICATION REPORT")
print("=" * 80)

# ------------------------------------------------------------
# 1. Classification Report
# ------------------------------------------------------------

vit_classification_report = classification_report(
    vit_test_true,
    vit_test_pred,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    zero_division=0
)

print(vit_classification_report)

# ------------------------------------------------------------
# 2. Overall Metrics
# ------------------------------------------------------------

vit_macro_precision = (
    precision_score(
        vit_test_true,
        vit_test_pred,
        labels=np.arange(len(class_names)),
        average="macro",
        zero_division=0
    ) * 100
)

vit_macro_recall = (
    recall_score(
        vit_test_true,
        vit_test_pred,
        labels=np.arange(len(class_names)),
        average="macro",
        zero_division=0
    ) * 100
)

vit_macro_f1 = (
    f1_score(
        vit_test_true,
        vit_test_pred,
        labels=np.arange(len(class_names)),
        average="macro",
        zero_division=0
    ) * 100
)

vit_weighted_precision = (
    precision_score(
        vit_test_true,
        vit_test_pred,
        labels=np.arange(len(class_names)),
        average="weighted",
        zero_division=0
    ) * 100
)

vit_weighted_recall = (
    recall_score(
        vit_test_true,
        vit_test_pred,
        labels=np.arange(len(class_names)),
        average="weighted",
        zero_division=0
    ) * 100
)

vit_weighted_f1 = (
    f1_score(
        vit_test_true,
        vit_test_pred,
        labels=np.arange(len(class_names)),
        average="weighted",
        zero_division=0
    ) * 100
)

# ------------------------------------------------------------
# 3. Overall Performance Summary
# ------------------------------------------------------------

print("=" * 80)
print("PURE ViT OVERALL PERFORMANCE")
print("=" * 80)

print(
    f"Test Accuracy          : {vit_test_accuracy:.2f}%"
)

print(
    f"Macro Precision        : {vit_macro_precision:.2f}%"
)

print(
    f"Macro Recall           : {vit_macro_recall:.2f}%"
)

print(
    f"Macro F1 Score         : {vit_macro_f1:.2f}%"
)

print(
    f"Weighted Precision     : {vit_weighted_precision:.2f}%"
)

print(
    f"Weighted Recall        : {vit_weighted_recall:.2f}%"
)

print(
    f"Weighted F1 Score      : {vit_weighted_f1:.2f}%"
)

print("=" * 80)
print("✓ PURE ViT CLASSIFICATION REPORT COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 101: PURE ViT-B/16 CONFUSION MATRIX
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

print("=" * 80)
print("PURE ViT-B/16 CONFUSION MATRIX")
print("=" * 80)

# ------------------------------------------------------------
# 1. Generate Confusion Matrix
# ------------------------------------------------------------

vit_cm = confusion_matrix(
    vit_test_true,
    vit_test_pred,
    labels=np.arange(len(class_names))
)

# ------------------------------------------------------------
# 2. Plot Confusion Matrix
# ------------------------------------------------------------

plt.figure(figsize=(20, 17))

sns.heatmap(
    vit_cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.3,
    linecolor="white"
)

plt.title(
    "Pure ViT-B/16 Confusion Matrix",
    fontsize=18,
    fontweight="bold",
    pad=15
)

plt.xlabel(
    "Predicted Class",
    fontsize=13,
    fontweight="bold"
)

plt.ylabel(
    "True Class",
    fontsize=13,
    fontweight="bold"
)

plt.xticks(
    rotation=90,
    fontsize=9
)

plt.yticks(
    rotation=0,
    fontsize=9
)

plt.tight_layout()

# ------------------------------------------------------------
# 3. Save Figure
# ------------------------------------------------------------

plt.savefig(
    "vit_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 4. Summary
# ------------------------------------------------------------

print(f"\nConfusion Matrix Shape: {vit_cm.shape}")

print(
    f"Total Test Samples: "
    f"{len(vit_test_true)}"
)

print(
    f"Correct Predictions: "
    f"{np.sum(vit_test_true == vit_test_pred)}"
)

print(
    f"Incorrect Predictions: "
    f"{np.sum(vit_test_true != vit_test_pred)}"
)

print("\n✓ Saved: vit_confusion_matrix.png")

print("=" * 80)
print("✓ PURE ViT CONFUSION MATRIX COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 102: PURE ViT MISCLASSIFIED IMAGES
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import torch

print("=" * 80)
print("PURE ViT-B/16 MISCLASSIFIED IMAGES")
print("=" * 80)

# ------------------------------------------------------------
# 1. Find Misclassified Samples
# ------------------------------------------------------------

vit_misclassified_indices = np.where(
    vit_test_true != vit_test_pred
)[0]

total_test = len(vit_test_true)
total_wrong = len(vit_misclassified_indices)
total_correct = total_test - total_wrong

print(f"Total Test Images       : {total_test}")
print(f"Correct Predictions     : {total_correct}")
print(f"Misclassified Images    : {total_wrong}")

# ------------------------------------------------------------
# 2. Display Misclassified Images
# ------------------------------------------------------------

if total_wrong == 0:

    print("\n🎉 No misclassified images found!")

else:

    num_to_show = min(
        20,
        total_wrong
    )

    cols = 4

    rows = int(
        np.ceil(
            num_to_show / cols
        )
    )

    plt.figure(
        figsize=(18, rows * 4)
    )

    for plot_idx, test_idx in enumerate(
        vit_misclassified_indices[:num_to_show]
    ):

        # ----------------------------------------------------
        # Get Image
        # ----------------------------------------------------

        image, true_label = test_dataset[
            test_idx
        ]

        pred_label = vit_test_pred[
            test_idx
        ]

        confidence = (
            vit_test_probs[test_idx][pred_label]
            * 100
        )

        # ----------------------------------------------------
        # Reverse ImageNet Normalization
        # ----------------------------------------------------

        image_display = image.clone()

        mean = torch.tensor(
            [0.485, 0.456, 0.406]
        ).view(3, 1, 1)

        std = torch.tensor(
            [0.229, 0.224, 0.225]
        ).view(3, 1, 1)

        image_display = (
            image_display * std + mean
        )

        image_display = torch.clamp(
            image_display,
            0,
            1
        )

        image_display = (
            image_display
            .permute(1, 2, 0)
            .cpu()
            .numpy()
        )

        # ----------------------------------------------------
        # Plot
        # ----------------------------------------------------

        ax = plt.subplot(
            rows,
            cols,
            plot_idx + 1
        )

        ax.imshow(
            image_display
        )

        ax.set_title(
            f"True: {class_names[true_label]}\n"
            f"Pred: {class_names[pred_label]}\n"
            f"Confidence: {confidence:.1f}%",
            fontsize=10
        )

        ax.axis("off")

    plt.suptitle(
        "Pure ViT-B/16 Misclassified Test Images",
        fontsize=18,
        fontweight="bold"
    )

    plt.tight_layout(
        rect=[0, 0, 1, 0.96]
    )

    plt.savefig(
        "vit_misclassified_images.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print(
        f"\n✓ Displayed {num_to_show} "
        f"of {total_wrong} misclassified images."
    )

    print(
        "✓ Saved: vit_misclassified_images.png"
    )

print("=" * 80)
print("✓ PURE ViT MISCLASSIFIED IMAGE ANALYSIS COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 103: PURE ViT-B/16 PREDICTION CONFIDENCE ANALYSIS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 80)
print("PURE ViT-B/16 PREDICTION CONFIDENCE ANALYSIS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Prediction Confidence
# ------------------------------------------------------------

vit_prediction_confidences = (
    np.max(
        vit_test_probs,
        axis=1
    ) * 100
)

# Correct / Incorrect masks
vit_correct_mask = (
    vit_test_true == vit_test_pred
)

vit_incorrect_mask = (
    vit_test_true != vit_test_pred
)

vit_correct_confidences = (
    vit_prediction_confidences[
        vit_correct_mask
    ]
)

vit_incorrect_confidences = (
    vit_prediction_confidences[
        vit_incorrect_mask
    ]
)

# ------------------------------------------------------------
# 2. Confidence Summary
# ------------------------------------------------------------

print("\nCONFIDENCE SUMMARY")
print("-" * 80)

print(
    f"Average Confidence (All)      : "
    f"{np.mean(vit_prediction_confidences):.2f}%"
)

print(
    f"Average Confidence (Correct)   : "
    f"{np.mean(vit_correct_confidences):.2f}%"
)

if len(vit_incorrect_confidences) > 0:

    print(
        f"Average Confidence (Incorrect) : "
        f"{np.mean(vit_incorrect_confidences):.2f}%"
    )

else:

    print(
        "Average Confidence (Incorrect) : N/A"
    )

print(
    f"Minimum Confidence             : "
    f"{np.min(vit_prediction_confidences):.2f}%"
)

print(
    f"Maximum Confidence             : "
    f"{np.max(vit_prediction_confidences):.2f}%"
)

# ------------------------------------------------------------
# 3. Confidence Distribution
# ------------------------------------------------------------

plt.figure(
    figsize=(12, 6)
)

plt.hist(
    vit_correct_confidences,
    bins=20,
    alpha=0.7,
    label="Correct Predictions"
)

if len(vit_incorrect_confidences) > 0:

    plt.hist(
        vit_incorrect_confidences,
        bins=20,
        alpha=0.7,
        label="Incorrect Predictions"
    )

plt.xlabel(
    "Prediction Confidence (%)",
    fontsize=12
)

plt.ylabel(
    "Number of Images",
    fontsize=12
)

plt.title(
    "Pure ViT-B/16 Prediction Confidence Distribution",
    fontsize=16,
    fontweight="bold"
)

plt.legend()

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "vit_prediction_confidence.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 4. High-Confidence Wrong Predictions
# ------------------------------------------------------------

if len(vit_incorrect_confidences) > 0:

    print("\nTOP HIGH-CONFIDENCE WRONG PREDICTIONS")
    print("-" * 80)

    wrong_indices = np.where(
        vit_incorrect_mask
    )[0]

    sorted_wrong_indices = wrong_indices[
        np.argsort(
            vit_prediction_confidences[
                wrong_indices
            ]
        )[::-1]
    ]

    top_n = min(
        10,
        len(sorted_wrong_indices)
    )

    for rank, idx in enumerate(
        sorted_wrong_indices[:top_n],
        start=1
    ):

        true_name = class_names[
            vit_test_true[idx]
        ]

        predicted_name = class_names[
            vit_test_pred[idx]
        ]

        confidence = (
            vit_prediction_confidences[idx]
        )

        print(
            f"{rank:2d}. "
            f"True: {true_name:<18} | "
            f"Predicted: {predicted_name:<18} | "
            f"Confidence: {confidence:.2f}%"
        )

# ------------------------------------------------------------
# 5. Save Confidence Data
# ------------------------------------------------------------

vit_confidence_df = pd.DataFrame({
    "index": np.arange(
        len(vit_test_true)
    ),
    "true_class": [
        class_names[x]
        for x in vit_test_true
    ],
    "predicted_class": [
        class_names[x]
        for x in vit_test_pred
    ],
    "confidence": vit_prediction_confidences,
    "correct": vit_correct_mask
})

vit_confidence_df.to_csv(
    "vit_prediction_confidence.csv",
    index=False
)

print("\n✓ Saved:")
print("  vit_prediction_confidence.png")
print("  vit_prediction_confidence.csv")

print("=" * 80)
print("✓ PURE ViT CONFIDENCE ANALYSIS COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 104: PURE ViT-B/16 CLASS-WISE F1 SCORE
# ============================================================

from sklearn.metrics import precision_recall_fscore_support
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

print("=" * 80)
print("PURE ViT-B/16 CLASS-WISE PERFORMANCE")
print("=" * 80)

# ------------------------------------------------------------
# 1. Calculate Class-wise Metrics
# ------------------------------------------------------------

vit_labels = np.arange(
    len(class_names)
)

vit_precision, vit_recall, vit_f1, vit_support = (
    precision_recall_fscore_support(
        vit_test_true,
        vit_test_pred,
        labels=vit_labels,
        zero_division=0
    )
)

# ------------------------------------------------------------
# 2. Create DataFrame
# ------------------------------------------------------------

vit_class_metrics = pd.DataFrame({
    "Species": class_names,
    "Precision": vit_precision * 100,
    "Recall": vit_recall * 100,
    "F1 Score": vit_f1 * 100,
    "Support": vit_support
})

# Sort by F1
vit_class_metrics_sorted = (
    vit_class_metrics
    .sort_values(
        "F1 Score",
        ascending=False
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Display Complete Table
# ------------------------------------------------------------

print("\nCLASS-WISE PERFORMANCE")
print("-" * 80)

print(
    vit_class_metrics_sorted.to_string(
        index=False,
        formatters={
            "Precision": "{:.2f}%".format,
            "Recall": "{:.2f}%".format,
            "F1 Score": "{:.2f}%".format
        }
    )
)

# ------------------------------------------------------------
# 4. F1 Score Visualization
# ------------------------------------------------------------

plt.figure(
    figsize=(16, 10)
)

bars = plt.barh(
    vit_class_metrics_sorted["Species"],
    vit_class_metrics_sorted["F1 Score"]
)

plt.xlabel(
    "F1 Score (%)",
    fontsize=12
)

plt.ylabel(
    "Species",
    fontsize=12
)

plt.title(
    "Pure ViT-B/16 Class-wise F1 Score",
    fontsize=16,
    fontweight="bold"
)

plt.xlim(
    0,
    105
)

# Add values
for bar, value in zip(
    bars,
    vit_class_metrics_sorted["F1 Score"]
):

    plt.text(
        value + 1,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.1f}%",
        va="center",
        fontsize=8
    )

plt.grid(
    axis="x",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "vit_classwise_f1.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 5. Save Class-wise Metrics
# ------------------------------------------------------------

vit_class_metrics.to_csv(
    "vit_classwise_metrics.csv",
    index=False
)

print("\n✓ Saved:")
print("  vit_classwise_f1.png")
print("  vit_classwise_metrics.csv")

print("=" * 80)
print("✓ PURE ViT CLASS-WISE F1 ANALYSIS COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 105: PURE ViT SUPPORT VS F1 ANALYSIS
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("=" * 80)
print("PURE ViT-B/16 SUPPORT VS F1 ANALYSIS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Prepare Data
# ------------------------------------------------------------

vit_support_f1_df = vit_class_metrics.copy()

# Error count
vit_support_f1_df["Errors"] = (
    vit_support_f1_df["Support"]
    * (1 - vit_support_f1_df["Recall"] / 100)
).round().astype(int)

# Error rate
vit_support_f1_df["Error Rate"] = np.where(
    vit_support_f1_df["Support"] > 0,
    (
        vit_support_f1_df["Errors"]
        / vit_support_f1_df["Support"]
    ) * 100,
    0
)

# ------------------------------------------------------------
# 2. Display Support vs F1 Table
# ------------------------------------------------------------

print("\nSPECIES SUPPORT VS F1")
print("-" * 80)

vit_support_display = (
    vit_support_f1_df[
        [
            "Species",
            "Support",
            "F1 Score",
            "Recall",
            "Errors",
            "Error Rate"
        ]
    ]
    .sort_values(
        "Support"
    )
    .reset_index(drop=True)
)

print(
    vit_support_display.to_string(
        index=False,
        formatters={
            "F1 Score": "{:.2f}%".format,
            "Recall": "{:.2f}%".format,
            "Error Rate": "{:.2f}%".format
        }
    )
)

# ------------------------------------------------------------
# 3. Support vs F1 Scatter Plot
# ------------------------------------------------------------

plt.figure(
    figsize=(14, 9)
)

plt.scatter(
    vit_support_f1_df["Support"],
    vit_support_f1_df["F1 Score"],
    s=80,
    alpha=0.75
)

# Add species names
for _, row in vit_support_f1_df.iterrows():

    plt.annotate(
        row["Species"],
        (
            row["Support"],
            row["F1 Score"]
        ),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8
    )

plt.xlabel(
    "Test Set Support (Number of Images)",
    fontsize=12
)

plt.ylabel(
    "F1 Score (%)",
    fontsize=12
)

plt.title(
    "Pure ViT-B/16: Test Support vs Class-wise F1 Score",
    fontsize=16,
    fontweight="bold"
)

plt.ylim(
    -5,
    105
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "vit_support_vs_f1.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 4. Identify Low-Support Species
# ------------------------------------------------------------

LOW_SUPPORT_THRESHOLD = 3

vit_low_support_df = vit_support_f1_df[
    vit_support_f1_df["Support"]
    <= LOW_SUPPORT_THRESHOLD
].sort_values(
    "Support"
)

print("\nLOW-SUPPORT SPECIES")
print("-" * 80)

if len(vit_low_support_df) > 0:

    print(
        vit_low_support_df[
            [
                "Species",
                "Support",
                "F1 Score",
                "Recall"
            ]
        ].to_string(
            index=False,
            formatters={
                "F1 Score": "{:.2f}%".format,
                "Recall": "{:.2f}%".format
            }
        )
    )

else:

    print(
        "No species found with support <= 3."
    )

# ------------------------------------------------------------
# 5. Save Analysis
# ------------------------------------------------------------

vit_support_f1_df.to_csv(
    "vit_support_vs_f1.csv",
    index=False
)

print("\n✓ Saved:")
print("  vit_support_vs_f1.png")
print("  vit_support_vs_f1.csv")

print("=" * 80)
print("✓ PURE ViT SUPPORT VS F1 ANALYSIS COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 106: PURE ViT FINAL PERFORMANCE SUMMARY
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("PURE ViT-B/16 FINAL PERFORMANCE SUMMARY")
print("=" * 80)

# ------------------------------------------------------------
# 1. Cross-Validation Results
# ------------------------------------------------------------

print("\nCROSS-VALIDATION PERFORMANCE")
print("-" * 80)

print(
    f"Mean CV Accuracy : {vit_cv_mean:.2f}%"
)

print(
    f"CV Std           : {vit_cv_std:.2f}%"
)

print(
    f"Minimum CV Acc   : {vit_cv_min:.2f}%"
)

print(
    f"Maximum CV Acc   : {vit_cv_max:.2f}%"
)

print(
    f"Best Fold        : "
    f"{int(best_fold_row['fold'])}"
)

print(
    f"Best Epoch       : "
    f"{int(best_fold_row['best_epoch'])}"
)

# ------------------------------------------------------------
# 2. Test Performance
# ------------------------------------------------------------

print("\nINDEPENDENT TEST PERFORMANCE")
print("-" * 80)

print(
    f"Test Accuracy        : "
    f"{vit_test_accuracy:.2f}%"
)

print(
    f"Macro Precision      : "
    f"{vit_macro_precision:.2f}%"
)

print(
    f"Macro Recall         : "
    f"{vit_macro_recall:.2f}%"
)

print(
    f"Macro F1             : "
    f"{vit_macro_f1:.2f}%"
)

print(
    f"Weighted Precision   : "
    f"{vit_weighted_precision:.2f}%"
)

print(
    f"Weighted Recall      : "
    f"{vit_weighted_recall:.2f}%"
)

print(
    f"Weighted F1         : "
    f"{vit_weighted_f1:.2f}%"
)

# ------------------------------------------------------------
# 3. Create Summary DataFrame
# ------------------------------------------------------------

vit_final_summary = pd.DataFrame({
    "Metric": [
        "Mean CV Accuracy",
        "CV Standard Deviation",
        "Minimum CV Accuracy",
        "Maximum CV Accuracy",
        "Test Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1 Score",
        "Weighted Precision",
        "Weighted Recall",
        "Weighted F1 Score"
    ],

    "Value": [
        vit_cv_mean,
        vit_cv_std,
        vit_cv_min,
        vit_cv_max,
        vit_test_accuracy,
        vit_macro_precision,
        vit_macro_recall,
        vit_macro_f1,
        vit_weighted_precision,
        vit_weighted_recall,
        vit_weighted_f1
    ]
})

# ------------------------------------------------------------
# 4. Display Summary Table
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL PURE ViT PERFORMANCE TABLE")
print("=" * 80)

summary_display = vit_final_summary.copy()

summary_display["Value"] = (
    summary_display["Value"]
    .map(lambda x: f"{x:.2f}%")
)

print(
    summary_display.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# 5. Model Information
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MODEL INFORMATION")
print("=" * 80)

print("Architecture        : Pure ViT-B/16")
print("Backbone            : ViT-B/16")
print("Pretrained          : ImageNet")
print("Backbone Frozen     : Yes")
print("LSTM                : No")
print(f"Number of Classes   : {NUM_CLASSES}")
print(f"Test Samples        : {len(vit_test_true)}")
print("CV Folds            : 4")
print("Maximum Epochs      : 150")
print("Early Stopping      : 10 epochs")
print("Optimizer           : AdamW")
print("Learning Rate       : 0.001")
print("Weight Decay        : 0.0001")
print("Batch Size          : 32")

# ------------------------------------------------------------
# 6. Save Summary
# ------------------------------------------------------------

vit_final_summary.to_csv(
    "vit_final_performance_summary.csv",
    index=False
)

print("\n✓ Saved:")
print("  vit_final_performance_summary.csv")

print("=" * 80)
print("✓ PURE ViT FINAL SUMMARY COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 107: PURE ViT-B/16 MODEL EXPORT
# ============================================================

import torch
import json
import os

print("=" * 80)
print("PURE ViT-B/16 MODEL EXPORT")
print("=" * 80)

# ------------------------------------------------------------
# 1. Save Model Weights
# ------------------------------------------------------------

vit_model_path = "final_vit_leaf_classifier.pth"

torch.save(
    final_vit_model.state_dict(),
    vit_model_path
)

print(
    f"✓ Model weights saved: {vit_model_path}"
)

# ------------------------------------------------------------
# 2. Save Class Names
# ------------------------------------------------------------

vit_class_names_path = "vit_class_names.json"

with open(
    vit_class_names_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        class_names,
        f,
        ensure_ascii=False,
        indent=4
    )

print(
    f"✓ Class names saved: {vit_class_names_path}"
)

# ------------------------------------------------------------
# 3. Model Information
# ------------------------------------------------------------

vit_model_info = {
    "model_name": "Pure Vision Transformer",
    "architecture": "ViT-B/16",
    "backbone": "ViT-B/16",
    "pretrained": "ImageNet",
    "backbone_frozen": True,
    "lstm_used": False,
    "num_classes": NUM_CLASSES,

    "total_parameters": sum(
        p.numel()
        for p in final_vit_model.parameters()
    ),

    "trainable_parameters": sum(
        p.numel()
        for p in final_vit_model.parameters()
        if p.requires_grad
    ),

    "device": str(device),

    "cv_folds": 4,
    "max_epochs": 150,
    "early_stopping_patience": 10,

    "optimizer": "AdamW",
    "learning_rate": 0.001,
    "weight_decay": 0.0001,
    "batch_size": 32
}

vit_model_info_path = "vit_model_info.json"

with open(
    vit_model_info_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        vit_model_info,
        f,
        indent=4
    )

print(
    f"✓ Model information saved: {vit_model_info_path}"
)

# ------------------------------------------------------------
# 4. Save Performance Information
# ------------------------------------------------------------

vit_performance_info = {
    "mean_cv_accuracy": float(vit_cv_mean),
    "cv_std": float(vit_cv_std),
    "cv_min_accuracy": float(vit_cv_min),
    "cv_max_accuracy": float(vit_cv_max),

    "test_accuracy": float(vit_test_accuracy),

    "macro_precision": float(vit_macro_precision),
    "macro_recall": float(vit_macro_recall),
    "macro_f1": float(vit_macro_f1),

    "weighted_precision": float(
        vit_weighted_precision
    ),

    "weighted_recall": float(
        vit_weighted_recall
    ),

    "weighted_f1": float(
        vit_weighted_f1
    )
}

vit_performance_path = "vit_performance.json"

with open(
    vit_performance_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        vit_performance_info,
        f,
        indent=4
    )

print(
    f"✓ Performance saved: {vit_performance_path}"
)

# ------------------------------------------------------------
# 5. Check Exported Files
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("EXPORTED FILES")
print("=" * 80)

exported_files = [
    vit_model_path,
    vit_class_names_path,
    vit_model_info_path,
    vit_performance_path,

    "vit_cv_results.csv",
    "vit_final_performance_summary.csv",
    "vit_classwise_metrics.csv",
    "vit_prediction_confidence.csv",

    "vit_confusion_matrix.png",
    "vit_misclassified_images.png",
    "vit_prediction_confidence.png",
    "vit_classwise_f1.png",
    "vit_support_vs_f1.png"
]

for file_name in exported_files:

    status = "✓" if os.path.exists(
        file_name
    ) else "✗"

    print(
        f"{status} {file_name}"
    )

print("\n" + "=" * 80)
print("✓ PURE ViT MODEL EXPORT COMPLETE")
print("=" * 80)


In [ ]:

import os
import json

SEARCH_DIRS = [
    OUTPUT_ROOT,
    os.path.join(OUTPUT_ROOT, "old_outputs"),
    DATASET_PARENT,
]

print("=" * 65)
print("CHECKPOINT VERIFICATION")
print("=" * 65)

for root_dir in SEARCH_DIRS:
    if not os.path.isdir(root_dir):
        continue

    for root, dirs, files in os.walk(root_dir):
        # Avoid scanning unnecessarily large directory trees
        dirs[:] = [
            d for d in dirs
            if d not in [".git", "__pycache__"]
        ]

        for filename in files:
            lower = filename.lower()

            if (
                lower.endswith((".pth", ".pt", ".ckpt"))
                and any(k in lower for k in [
                    "efficientnet", "vit", "hybrid"
                ])
            ):
                print(os.path.join(root, filename))

print("\nCLASS-NAME FILES")
for root_dir in SEARCH_DIRS:
    if not os.path.isdir(root_dir):
        continue

    for root, dirs, files in os.walk(root_dir):
        dirs[:] = [
            d for d in dirs
            if d not in [".git", "__pycache__"]
        ]

        for filename in files:
            if filename.lower() == "class_names.json":
                print(os.path.join(root, filename))


In [ ]:

# ============================================================
# HYBRID MODEL | SEGMENT 1: LOAD TRAINED CHECKPOINTS
# EfficientNet-B0 + Pure ViT-B/16
# ============================================================

import os
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = len(class_names)

# ------------------------------------------------------------
# 1. Locate EfficientNet checkpoint
# ------------------------------------------------------------

EFFNET_PATH = (
    os.path.join(OUTPUT_ROOT, "old_outputs", "final_efficientnet_b0_leaf_classifier.pth")
)

if not os.path.exists(EFFNET_PATH):
    raise FileNotFoundError(
        f"EfficientNet checkpoint not found: {EFFNET_PATH}\n"
        "Update EFFNET_PATH to the actual checkpoint location."
    )

# ------------------------------------------------------------
# 2. Load EfficientNet-B0 trained weights
# ------------------------------------------------------------

effnet_model = models.efficientnet_b0(weights=None)
effnet_model.classifier[1] = nn.Linear(
    effnet_model.classifier[1].in_features,
    NUM_CLASSES
)

effnet_checkpoint = torch.load(
    EFFNET_PATH,
    map_location="cpu",
    weights_only=True
)

# Support either a raw state_dict or a wrapped checkpoint
if isinstance(effnet_checkpoint, dict):
    for key in ["state_dict", "model_state_dict", "model"]:
        if key in effnet_checkpoint and isinstance(
            effnet_checkpoint[key], dict
        ):
            effnet_checkpoint = effnet_checkpoint[key]
            break

# Remove DataParallel prefix if present
effnet_checkpoint = {
    k.replace("module.", "", 1) if k.startswith("module.") else k: v
    for k, v in effnet_checkpoint.items()
}

effnet_model.load_state_dict(effnet_checkpoint, strict=True)
effnet_model = effnet_model.to(device).eval()

# ------------------------------------------------------------
# 3. Load current notebook's trained ViT weights
# ------------------------------------------------------------

if "global_best_state" not in globals() or global_best_state is None:
    raise RuntimeError(
        "global_best_state পাওয়া যায়নি। "
        "ViT training-এর best checkpoint variable আগে তৈরি করো।"
    )

vit_model = models.vit_b_16(weights=None)
vit_model.heads.head = nn.Linear(
    vit_model.heads.head.in_features,
    NUM_CLASSES
)

vit_model.load_state_dict(global_best_state, strict=True)
vit_model = vit_model.to(device).eval()

# ------------------------------------------------------------
# 4. Freeze both pretrained/trained backbones
# ------------------------------------------------------------

for param in effnet_model.parameters():
    param.requires_grad = False

for param in vit_model.parameters():
    param.requires_grad = False

print("=" * 60)
print("CHECKPOINT LOADING SUCCESSFUL")
print("=" * 60)
print("Device:", device)
print("Number of classes:", NUM_CLASSES)
print("EfficientNet checkpoint:", EFFNET_PATH)
print("ViT weights: global_best_state from current notebook")
print("EfficientNet parameters frozen:", all(
    not p.requires_grad for p in effnet_model.parameters()
))
print("ViT parameters frozen:", all(
    not p.requires_grad for p in vit_model.parameters()
))


In [ ]:

# ============================================================
# HYBRID MODEL | SEGMENT 2: FEATURE EXTRACTION + FUSION
# EfficientNet-B0 (1280) + ViT-B/16 (768) = 2048
# ============================================================

import torch
import torch.nn as nn

# ------------------------------------------------------------
# 1. Feature Extractors
# ------------------------------------------------------------

class EfficientNetFeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.features = model.features
        self.avgpool = model.avgpool

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        return torch.flatten(x, 1)


class ViTFeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.conv_proj = model.conv_proj
        self.class_token = model.class_token
        self.encoder = model.encoder

    def forward(self, x):
        # ViT patch embedding
        x = self.conv_proj(x)
        x = x.reshape(x.shape[0], x.shape[1], -1)
        x = x.permute(0, 2, 1)

        # Add class token
        batch_size = x.shape[0]
        class_token = self.class_token.expand(batch_size, -1, -1)
        x = torch.cat([class_token, x], dim=1)

        # Positional embedding + transformer encoder
        x = self.encoder(x)

        # CLS token representation
        return x[:, 0]


effnet_features = EfficientNetFeatureExtractor(
    effnet_model
).to(device).eval()

vit_features = ViTFeatureExtractor(
    vit_model
).to(device).eval()

# Freeze feature extractors
for p in effnet_features.parameters():
    p.requires_grad = False

for p in vit_features.parameters():
    p.requires_grad = False

print("EfficientNet feature extractor ready")
print("ViT feature extractor ready")

# ------------------------------------------------------------
# 2. Test with a dummy input
# ------------------------------------------------------------

dummy_input = torch.randn(2, 3, 224, 224).to(device)

with torch.no_grad():
    eff_features = effnet_features(dummy_input)
    vit_features_out = vit_features(dummy_input)

    fused_features = torch.cat(
        [eff_features, vit_features_out], dim=1
    )

print("\nFeature shape verification")
print("EfficientNet features:", tuple(eff_features.shape))
print("ViT features:", tuple(vit_features_out.shape))
print("Fused features:", tuple(fused_features.shape))

assert eff_features.shape[1] == 1280
assert vit_features_out.shape[1] == 768
assert fused_features.shape[1] == 2048

print("\nSEGMENT 2 COMPLETE: Feature fusion verified!")


In [ ]:

# ============================================================
# HYBRID MODEL | SEGMENT 3: CLASSIFIER + TRAINING PREPARATION
# ============================================================

import torch
import torch.nn as nn

class EfficientNetViTHybrid(nn.Module):
    def __init__(
        self,
        effnet_features,
        vit_features,
        num_classes,
        dropout=0.3
    ):
        super().__init__()

        self.effnet_features = effnet_features
        self.vit_features = vit_features

        # Freeze both pretrained backbones
        for param in self.effnet_features.parameters():
            param.requires_grad = False

        for param in self.vit_features.parameters():
            param.requires_grad = False

        # Feature fusion: 1280 + 768 = 2048
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # Extract features without calculating backbone gradients
        with torch.no_grad():
            eff_features = self.effnet_features(x)
            vit_features = self.vit_features(x)

        # Concatenate both feature vectors
        fused_features = torch.cat(
            [eff_features, vit_features],
            dim=1
        )

        return self.classifier(fused_features)


# ------------------------------------------------------------
# Create Hybrid Model
# ------------------------------------------------------------

hybrid_model = EfficientNetViTHybrid(
    effnet_features=effnet_features,
    vit_features=vit_features,
    num_classes=NUM_CLASSES
).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    hybrid_model.classifier.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

# Learning-rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

print("=" * 60)
print("HYBRID MODEL CREATED")
print("=" * 60)
print("Architecture: EfficientNet-B0 + ViT-B/16")
print("Fusion dimensions: 1280 + 768 = 2048")
print("Output classes:", NUM_CLASSES)
print("Trainable parameters:", sum(
    p.numel()
    for p in hybrid_model.parameters()
    if p.requires_grad
))
print("Device:", device)
print("Both backbones frozen: Yes")
print("Classifier optimizer: AdamW")
print("\nSEGMENT 3 COMPLETE!")


In [ ]:

# ============================================================
# HYBRID MODEL | SEGMENT 4: 4-FOLD DATA PREPARATION
# ============================================================

import numpy as np
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

NUM_FOLDS = 4
MAX_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 7
BATCH_SIZE = 16
NUM_WORKERS = 2

# ------------------------------------------------------------
# Verify dataset exists
# ------------------------------------------------------------

if "train_dataset" not in globals():
    raise NameError(
        "train_dataset পাওয়া যায়নি। "
        "আগে dataset loading cell run করতে হবে।"
    )

if len(train_dataset) == 0:
    raise ValueError("train_dataset is empty!")

# ------------------------------------------------------------
# Extract labels
# ------------------------------------------------------------

all_labels = []

for i in range(len(train_dataset)):
    label = train_dataset[i][1]

    if torch.is_tensor(label):
        label = label.item()

    all_labels.append(int(label))

all_labels = np.asarray(all_labels)

print("Training samples:", len(all_labels))
print("Classes found:", len(np.unique(all_labels)))
print("Configured classes:", NUM_CLASSES)

assert len(np.unique(all_labels)) == NUM_CLASSES, (
    "Dataset label count and NUM_CLASSES do not match."
)

# ------------------------------------------------------------
# Stratified 4-fold split
# ------------------------------------------------------------

skf = StratifiedKFold(
    n_splits=NUM_FOLDS,
    shuffle=True,
    random_state=42
)

fold_splits = list(
    skf.split(np.zeros(len(all_labels)), all_labels)
)

for fold, (train_idx, val_idx) in enumerate(
    fold_splits, start=1
):
    print(
        f"Fold {fold}: "
        f"Train = {len(train_idx)}, "
        f"Validation = {len(val_idx)}"
    )

print("\nSEGMENT 4 COMPLETE!")
print("Fold splits are ready.")


In [ ]:
##Hybrid Model Training (4-Fold Cross Validation)
import copy
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# Keep frozen feature extractors in evaluation mode
class EfficientNetViTHybrid(EfficientNetViTHybrid):
    def train(self, mode=True):
        super().train(mode)
        self.effnet_features.eval()
        self.vit_features.eval()
        return self

os.makedirs(os.path.join(OUTPUT_ROOT, "hybrid_checkpoints"), exist_ok=True)

fold_results = []
best_overall_f1 = -1.0

for fold, (train_idx, val_idx) in enumerate(fold_splits, start=1):
    print(f"\n{'='*15} Fold {fold}/{NUM_FOLDS} {'='*15}")

    train_loader = DataLoader(
        Subset(train_dataset, train_idx),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda")
    )

    val_loader = DataLoader(
        Subset(train_dataset, val_idx),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda")
    )

    hybrid_model = EfficientNetViTHybrid(
        effnet_features, vit_features, NUM_CLASSES
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        hybrid_model.classifier.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3
    )

    best_fold_f1 = -1.0
    best_fold_loss = float("inf")
    best_fold_state = None
    patience_counter = 0

    for epoch in range(MAX_EPOCHS):
        # Training
        hybrid_model.train()
        train_loss_sum = 0.0
        train_count = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device).long()

            optimizer.zero_grad(set_to_none=True)
            outputs = hybrid_model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * labels.size(0)
            train_count += labels.size(0)

        train_loss = train_loss_sum / max(train_count, 1)

        # Validation
        hybrid_model.eval()
        val_loss_sum = 0.0
        val_count = 0
        y_true, y_pred = [], []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device).long()

                outputs = hybrid_model(images)
                loss = criterion(outputs, labels)

                val_loss_sum += loss.item() * labels.size(0)
                val_count += labels.size(0)

                predictions = outputs.argmax(dim=1)
                y_true.extend(labels.cpu().numpy())
                y_pred.extend(predictions.cpu().numpy())

        val_loss = val_loss_sum / max(val_count, 1)
        val_acc = accuracy_score(y_true, y_pred)
        val_f1 = f1_score(
            y_true, y_pred, average="macro", zero_division=0
        )

        scheduler.step(val_loss)

        if val_f1 > best_fold_f1:
            best_fold_f1 = val_f1
            best_fold_loss = val_loss
            best_fold_state = copy.deepcopy(hybrid_model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(
                f"Epoch {epoch+1:02d}/{MAX_EPOCHS} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Val Loss: {val_loss:.4f} | "
                f"Val Acc: {val_acc:.4f} | "
                f"Macro F1: {val_f1:.4f}"
            )

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

    # Save best checkpoint for this fold
    fold_path = os.path.join(
        OUTPUT_ROOT,
        "hybrid_checkpoints",
        f"hybrid_fold_{fold}.pth"
    )
    torch.save(best_fold_state, fold_path)

    fold_results.append({
        "fold": fold,
        "best_macro_f1": best_fold_f1,
        "best_val_loss": best_fold_loss
    })

    print(f"Fold {fold} best Macro F1: {best_fold_f1:.4f}")
    print(f"Saved: {fold_path}")

    if best_fold_f1 > best_overall_f1:
        best_overall_f1 = best_fold_f1
        torch.save(
            best_fold_state,
            os.path.join(OUTPUT_ROOT, "best_hybrid.pth")
        )

print("\n========== CROSS-VALIDATION RESULTS ==========")
for result in fold_results:
    print(
        f"Fold {result['fold']}: "
        f"Macro F1 = {result['best_macro_f1']:.4f}"
    )

print(
    f"\nMean CV Macro F1: "
    f"{np.mean([r['best_macro_f1'] for r in fold_results]):.4f}"
)
print(f"Best fold Macro F1: {best_overall_f1:.4f}")
print("Best checkpoint:", os.path.join(OUTPUT_ROOT, "best_hybrid.pth"))

In [ ]:
## Hybrid Model Test Evaluation 📊
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

CHECKPOINT_PATH = os.path.join(OUTPUT_ROOT, "best_hybrid.pth")

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(
        "best_hybrid.pth paoa jayni. Segment 5 successfully run hoyeche kina check koro."
    )

if "test_dataset" not in globals():
    raise NameError(
        "test_dataset define kora nei. Tomar existing test dataset variable check korte hobe."
    )

test_loader = DataLoader(
    test_dataset,
    batch_size= BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

# Rebuild model and load best checkpoint
hybrid_model = EfficientNetViTHybrid(
    effnet_features, vit_features, NUM_CLASSES
).to(device)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=True
)
hybrid_model.load_state_dict(checkpoint)
hybrid_model.eval()

y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = hybrid_model(images)
        predictions = outputs.argmax(dim=1)

        y_true.extend(labels.numpy().tolist())
        y_pred.extend(predictions.cpu().numpy().tolist())

# Metrics
test_accuracy = accuracy_score(y_true, y_pred)
test_precision = precision_score(
    y_true, y_pred, average="macro", zero_division=0
)
test_recall = recall_score(
    y_true, y_pred, average="macro", zero_division=0
)
test_f1 = f1_score(
    y_true, y_pred, average="macro", zero_division=0
)

print("========== HYBRID TEST RESULTS ==========")
print(f"Accuracy        : {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Macro Precision : {test_precision:.4f}")
print(f"Macro Recall    : {test_recall:.4f}")
print(f"Macro F1-score  : {test_f1:.4f}")

print("\n========== CLASSIFICATION REPORT ==========")
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(NUM_CLASSES)),
    target_names=[str(c) for c in class_names],
    zero_division=0
))

# Confusion Matrix
cm = confusion_matrix(
    y_true, y_pred, labels=list(range(NUM_CLASSES))
)

plt.figure(figsize=(14, 12))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Hybrid EfficientNet-B0 + ViT-B/16 Confusion Matrix")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_ROOT, "hybrid_confusion_matrix.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("Confusion matrix saved successfully!")

In [ ]:
## Save Hybrid Results + Model Comparison 📊
import os
import json
import pandas as pd
import numpy as np

OUTPUT_DIR = os.path.join(OUTPUT_ROOT, "hybrid_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Save Hybrid metrics
hybrid_metrics = {
    "model": "EfficientNet-B0 + ViT-B/16 Hybrid",
    "test_accuracy": float(test_accuracy),
    "macro_precision": float(test_precision),
    "macro_recall": float(test_recall),
    "macro_f1_score": float(test_f1),
    "mean_cv_macro_f1": float(
        np.mean([r["best_macro_f1"] for r in fold_results])
    ),
    "best_cv_macro_f1": float(best_overall_f1),
    "num_classes": int(NUM_CLASSES),
    "test_samples": int(len(y_true))
}

with open(
    os.path.join(OUTPUT_DIR, "hybrid_metrics.json"),
    "w"
) as f:
    json.dump(hybrid_metrics, f, indent=4)

pd.DataFrame([hybrid_metrics]).to_csv(
    os.path.join(OUTPUT_DIR, "hybrid_metrics.csv"),
    index=False
)

# 2. Save fold-wise CV results
pd.DataFrame(fold_results).to_csv(
    os.path.join(OUTPUT_DIR, "hybrid_cv_results.csv"),
    index=False
)

# 3. Save classification report
report = classification_report(
    y_true,
    y_pred,
    labels=list(range(NUM_CLASSES)),
    target_names=[str(c) for c in class_names],
    output_dict=True,
    zero_division=0
)

pd.DataFrame(report).transpose().to_csv(
    os.path.join(OUTPUT_DIR, "hybrid_classification_report.csv")
)

# 4. Compare with previously reported model results
# Replace these values with the final test results from each model's
# corresponding evaluation output before using this table in a paper.
comparison = [
    {"Model": "EfficientNet-B0", "Test Accuracy (%)": 93.22},
    {"Model": "ResNet50", "Test Accuracy (%)": 90.96},
    {"Model": "DenseNet121", "Test Accuracy (%)": 89.83},
    {"Model": "ViT + LSTM", "Test Accuracy (%)": 87.01},
    {"Model": "Pure ViT-B/16", "Test Accuracy (%)": 93.22},
    {
        "Model": "Hybrid EfficientNet-B0 + ViT-B/16",
        "Test Accuracy (%)": round(test_accuracy * 100, 2)
    }
]

comparison_df = pd.DataFrame(comparison)

comparison_df.to_csv(
    os.path.join(OUTPUT_DIR, "model_comparison.csv"),
    index=False
)

print("========== HYBRID METRICS ==========")
print(pd.DataFrame([hybrid_metrics]).to_string(index=False))

print("\n========== MODEL COMPARISON ==========")
print(comparison_df.to_string(index=False))

print(f"\nAll result files saved in: {OUTPUT_DIR}")

In [ ]:
## Model Comparison Graph 📈
import os
import matplotlib.pyplot as plt
import pandas as pd

OUTPUT_DIR = os.path.join(OUTPUT_ROOT, "hybrid_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load comparison results
comparison_df = pd.read_csv(
    os.path.join(OUTPUT_DIR, "model_comparison.csv")
)

plt.figure(figsize=(11, 6))

bars = plt.bar(
    comparison_df["Model"],
    comparison_df["Test Accuracy (%)"]
)

plt.xlabel("Models")
plt.ylabel("Test Accuracy (%)")
plt.title("Performance Comparison of Leaf Classification Models")
plt.ylim(0, 105)
plt.xticks(rotation=25, ha="right")

for bar, value in zip(bars, comparison_df["Test Accuracy (%)"]):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{value:.2f}%",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()

graph_path = os.path.join(
    OUTPUT_DIR, "model_accuracy_comparison.png"
)
plt.savefig(graph_path, dpi=300, bbox_inches="tight")
plt.show()

print("Graph saved:", graph_path)

In [ ]:
## Final Check + Package All Hybrid Outputs
import os
import zipfile

WORK_DIR = OUTPUT_ROOT
RESULTS_DIR = os.path.join(WORK_DIR, "hybrid_results")
CHECKPOINT_DIR = os.path.join(WORK_DIR, "hybrid_checkpoints")
ZIP_PATH = os.path.join(WORK_DIR, "hybrid_leaf_project.zip")

# Required files check
required_files = [
    os.path.join(WORK_DIR, "best_hybrid.pth"),
    os.path.join(RESULTS_DIR, "hybrid_metrics.json"),
    os.path.join(RESULTS_DIR, "hybrid_metrics.csv"),
    os.path.join(RESULTS_DIR, "hybrid_cv_results.csv"),
    os.path.join(RESULTS_DIR, "model_comparison.csv"),
    os.path.join(RESULTS_DIR, "model_accuracy_comparison.png"),
    os.path.join(WORK_DIR, "hybrid_confusion_matrix.png"),
]

print("========== FILE CHECK ==========")
for path in required_files:
    print(("✅ " if os.path.exists(path) else "❌ "), path)

missing = [p for p in required_files if not os.path.exists(p)]

if missing:
    print("\nSome files are missing. Check previous segments before packaging.")
else:
    with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:

        # Include hybrid checkpoint
        zf.write(
            os.path.join(WORK_DIR, "best_hybrid.pth"),
            arcname="best_hybrid.pth"
        )

        # Include results
        for root, _, files in os.walk(RESULTS_DIR):
            for filename in files:
                full_path = os.path.join(root, filename)
                zf.write(
                    full_path,
                    arcname=os.path.relpath(full_path, WORK_DIR)
                )

        # Include confusion matrix
        confusion_path = os.path.join(
            WORK_DIR, "hybrid_confusion_matrix.png"
        )
        if os.path.exists(confusion_path):
            zf.write(
                confusion_path,
                arcname="hybrid_confusion_matrix.png"
            )

        # Include fold checkpoints
        if os.path.isdir(CHECKPOINT_DIR):
            for filename in os.listdir(CHECKPOINT_DIR):
                full_path = os.path.join(CHECKPOINT_DIR, filename)
                if os.path.isfile(full_path):
                    zf.write(
                        full_path,
                        arcname=os.path.relpath(full_path, WORK_DIR)
                    )

    print("\n✅ ZIP created:", ZIP_PATH)
    print("ZIP size:", round(os.path.getsize(ZIP_PATH) / (1024**2), 2), "MB")